# Answer-flip study — replication on Qwen3-VL-8B (Run-All)

A secondary analysis from the study plan: the same measures on a third model, from a different family, so H1, H2a, H2b and H3 can be checked for replication. Each run handles **one** benchmark, chosen in the config cell.

| `BENCHMARK` | Questions | Expected time | Videos downloaded |
|---|---|---|---|
| `"videomme"` | 300 (100 videos) | 1.5–2 h | 5 × 5 GB zips |
| `"egoschema"` | 300 | 3–4 h | 5 × 21 GB zips (only the needed videos are kept) |
| `"longvideobench"` | 300 (75 per duration group) | 3–4 h | 31 × 5 GB parts, streamed |

**Model.** Qwen3-VL-8B-Instruct in FlashVID's evaluation setting: 32 frames (16 temporal groups of 2 frames, each with a timestamp), at most 448 × 448 pixels per frame (FlashVID's baseline script), 3-D positions and DeepStack features handled exactly as in the official model.

**Methods.** FlashVID (official Qwen3-VL settings: 4 minimum segments, ADTS v1, pruning to 10% at layer 28), uniform pooling and random drop at 5%, 10% and 25%, plus the same four harmless noise passes as the main study.

**Two safety checks stop the run before wasting GPU hours:** (1) our pipeline vs the official processor + model forward, (2) cached vs uncached answers on the first video.

## How to run
1. Settings → Accelerator **GPU T4 x2**, Internet **On**, HF_TOKEN in Add-ons → Secrets (already set).
2. `BENCHMARK = "videomme"`, `QUICK_TEST = True` → **Run All** (~30 min, uses the real model on 3 questions). Check: `Compressor self-test OK ✅`, equivalence check `PASS`, `Cache check: PASS`, a few ✓ lines.
3. Then `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**, one benchmark per run (three runs).
4. If a run stops at the 12-hour limit: **Add Input** → that version's output, and run again. It continues where it stopped.
5. Send back from `flips_rep_qwen3vl_<benchmark>/`: `summary_<benchmark>.json`, `cells.csv`, `summary_<benchmark>.png`, `results.jsonl`, `cache_check.json`, `equivalence_check.json`, `manifest_<benchmark>.json`, `run_record_*.json` (+ `errors.log` if not empty).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — change only BENCHMARK and QUICK_TEST                    ║
# ╚══════════════════════════════════════════════════════════════════╝
BENCHMARK  = "videomme"   # one of: "videomme", "egoschema", "longvideobench"
QUICK_TEST = True         # True = ~30-min smoke test (real model, 4 questions). False = the real run.

MODEL = "qwen3vl"       # fixed for this notebook: "llava_video" or "qwen3vl"

# ── Replication sample: 300 questions per benchmark, drawn once (seed 20261015) from the main study's manifests.
#    Video-MME: 100 of the 213 confirmatory videos (all their questions); EgoSchema: 300 of 500;
#    LongVideoBench: 75 per duration group. The question IDs are fixed in REP_MANIFEST below (with their hash).
SAMPLE_SEED = 20261015
VIDEOMME_CONFIRM_CHUNKS = ["02", "08", "16", "18", "20"]
LVB_PER_GROUP = 125
MVBENCH_PER_TASK, MLVU_PER_TASK = 25, 72          # not used here (kept so the shared data code runs unchanged)
PREREG_URL = "https://claude.ai/code/artifact/65cb336e-3916-4101-9914-8dd1ca98e189"

if MODEL == "llava_video":
    MODEL_ID = "lmms-lab/LLaVA-Video-7B-Qwen2"          # original weights, converted to the transformers format below
    HF_BASE  = "llava-hf/llava-onevision-qwen2-7b-ov-hf"  # same architecture; provides the transformers config
    NUM_FRAMES = 64                                     # FlashVID's LLaVA-Video setting (scripts/llava_vid.sh)
    METHOD_GRID = {"flashvid": [0.05, 0.10, 0.25], "holitom": [0.05, 0.10, 0.25], "visionzip": [0.05, 0.10, 0.25],
                   "uniform": [0.05, 0.10, 0.25], "random": [0.05, 0.10, 0.25]}
    FLASHVID = dict(alpha=0.7, token_selection_method="attn_div_v2", temporal_threshold=0.8, do_segment=True,
                    segment_threshold=0.9, min_segment_num=8, complementary_segment=True,
                    expansion=1.25, pruning_layer=20, llm_retention_ratio=0.3)
    HOLITOM = dict(T={0.05: 0.65, 0.10: 0.65, 0.25: 0.80}, beta=0.6, D=0.0, K=7, max_window=1024,
                   inner_layer=18, inner_r=0.5)
else:
    MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
    NUM_FRAMES = 32                                     # FlashVID's Qwen3-VL setting (scripts/qwen3_vl.sh)
    MIN_PIXELS, MAX_PIXELS = 50716, 200704              # per frame, as in FlashVID's Qwen3-VL baseline script
    METHOD_GRID = {"flashvid": [0.05, 0.10, 0.25], "uniform": [0.05, 0.10, 0.25], "random": [0.05, 0.10, 0.25]}
    FLASHVID = dict(alpha=0.7, token_selection_method="attn_div", temporal_threshold=0.8, do_segment=True,
                    segment_threshold=0.9, min_segment_num=4, complementary_segment=True,
                    expansion=1.25, pruning_layer=28, llm_retention_ratio=0.1)

# ── Four noise passes of the full model (as in the main study) ──
NOISE_FRAME_SHIFTS = {"frame_shift": 0.5, "frame_shift_q": 0.25}   # fraction of one sampling interval
NOISE_OPTION_ORDERS = ["option_cycle", "option_reverse"]

SEED = 0
MAX_RUNTIME_HOURS = 11.0
DATA_CHECK_ONLY = False
OUT_DIR  = f"/kaggle/working/flips_rep_{MODEL}_{BENCHMARK}"
WORK_DIR = "/tmp/astra_data"
QUICK_MAX_Q = 4 if QUICK_TEST else None
if QUICK_TEST:
    OUT_DIR = OUT_DIR + "_quick"
assert BENCHMARK in ("videomme", "egoschema", "longvideobench"), f"Unknown BENCHMARK {BENCHMARK}"
print("Model:", MODEL_ID, "| benchmark:", BENCHMARK, "| mode:", "QUICK TEST" if QUICK_TEST else "REAL RUN",
      "| frames:", NUM_FRAMES)

In [ ]:
# ── Frozen replication sample (question IDs per benchmark, with their SHA-256) ──
# Drawn once with seed 20261015 from the main study's question lists: Video-MME 100 of the 213 videos (all questions),
# EgoSchema 300 of 500 questions, LongVideoBench 75 questions per duration group.
REP_MANIFEST = {
    "videomme": {"sha256": "1f816d6cffb0ebb60c72f3e5252d25add7ecc6e58d6e5b1a7f71c0da45085977", "qids": ["014-1", "014-2", "014-3", "026-1", "026-2", "026-3", "032-1", "032-2", "032-3", "038-1", "038-2", "038-3", "066-1", "066-2", "066-3", "075-1", "075-2", "075-3", "126-1", "126-2", "126-3", "133-1", "133-2", "133-3", "163-1", "163-2", "163-3", "165-1", "165-2", "165-3", "180-1", "180-2", "180-3", "182-1", "182-2", "182-3", "188-1", "188-2", "188-3", "206-1", "206-2", "206-3", "210-1", "210-2", "210-3", "219-1", "219-2", "219-3", "224-1", "224-2", "224-3", "228-1", "228-2", "228-3", "240-1", "240-2", "240-3", "246-1", "246-2", "246-3", "258-1", "258-2", "258-3", "260-1", "260-2", "260-3", "266-1", "266-2", "266-3", "269-1", "269-2", "269-3", "270-1", "270-2", "270-3", "275-1", "275-2", "275-3", "283-1", "283-2", "283-3", "298-1", "298-2", "298-3", "323-1", "323-2", "323-3", "329-1", "329-2", "329-3", "332-1", "332-2", "332-3", "334-1", "334-2", "334-3", "335-1", "335-2", "335-3", "344-1", "344-2", "344-3", "349-1", "349-2", "349-3", "352-1", "352-2", "352-3", "355-1", "355-2", "355-3", "361-1", "361-2", "361-3", "362-1", "362-2", "362-3", "370-1", "370-2", "370-3", "380-1", "380-2", "380-3", "390-1", "390-2", "390-3", "394-1", "394-2", "394-3", "402-1", "402-2", "402-3", "410-1", "410-2", "410-3", "412-1", "412-2", "412-3", "419-1", "419-2", "419-3", "428-1", "428-2", "428-3", "429-1", "429-2", "429-3", "433-1", "433-2", "433-3", "437-1", "437-2", "437-3", "447-1", "447-2", "447-3", "449-1", "449-2", "449-3", "452-1", "452-2", "452-3", "456-1", "456-2", "456-3", "461-1", "461-2", "461-3", "463-1", "463-2", "463-3", "465-1", "465-2", "465-3", "466-1", "466-2", "466-3", "499-1", "499-2", "499-3", "502-1", "502-2", "502-3", "506-1", "506-2", "506-3", "507-1", "507-2", "507-3", "527-1", "527-2", "527-3", "534-1", "534-2", "534-3", "538-1", "538-2", "538-3", "547-1", "547-2", "547-3", "555-1", "555-2", "555-3", "556-1", "556-2", "556-3", "586-1", "586-2", "586-3", "606-1", "606-2", "606-3", "623-1", "623-2", "623-3", "674-1", "674-2", "674-3", "680-1", "680-2", "680-3", "705-1", "705-2", "705-3", "709-1", "709-2", "709-3", "713-1", "713-2", "713-3", "722-1", "722-2", "722-3", "724-1", "724-2", "724-3", "726-1", "726-2", "726-3", "744-1", "744-2", "744-3", "747-1", "747-2", "747-3", "761-1", "761-2", "761-3", "762-1", "762-2", "762-3", "768-1", "768-2", "768-3", "774-1", "774-2", "774-3", "779-1", "779-2", "779-3", "784-1", "784-2", "784-3", "794-1", "794-2", "794-3", "802-1", "802-2", "802-3", "838-1", "838-2", "838-3", "840-1", "840-2", "840-3", "856-1", "856-2", "856-3", "859-1", "859-2", "859-3", "862-1", "862-2", "862-3", "867-1", "867-2", "867-3", "871-1", "871-2", "871-3", "875-1", "875-2", "875-3", "879-1", "879-2", "879-3", "894-1", "894-2", "894-3"]},
    "egoschema": {"sha256": "c230cd8256575f5b036bd3ddfaa1bc85e4fd740058c71caf0e926db77b0f9469", "qids": ["00000", "00001", "00002", "00003", "00004", "00005", "00006", "00007", "00008", "00009", "00010", "00011", "00013", "00014", "00017", "00019", "00021", "00024", "00026", "00027", "00028", "00029", "00030", "00031", "00032", "00033", "00035", "00036", "00038", "00039", "00040", "00041", "00042", "00043", "00044", "00046", "00049", "00053", "00054", "00056", "00058", "00059", "00061", "00064", "00065", "00066", "00068", "00069", "00071", "00072", "00073", "00074", "00075", "00076", "00077", "00079", "00080", "00083", "00084", "00085", "00087", "00088", "00092", "00093", "00095", "00096", "00097", "00098", "00099", "00101", "00102", "00104", "00105", "00106", "00107", "00114", "00115", "00116", "00117", "00119", "00120", "00121", "00123", "00124", "00125", "00126", "00127", "00128", "00130", "00134", "00135", "00136", "00138", "00140", "00141", "00142", "00143", "00144", "00145", "00146", "00149", "00150", "00151", "00152", "00153", "00154", "00155", "00156", "00159", "00160", "00161", "00162", "00164", "00166", "00167", "00168", "00172", "00173", "00174", "00176", "00178", "00182", "00187", "00188", "00189", "00190", "00191", "00192", "00194", "00200", "00201", "00203", "00204", "00206", "00209", "00210", "00211", "00212", "00214", "00220", "00222", "00223", "00227", "00228", "00231", "00232", "00234", "00235", "00237", "00238", "00239", "00240", "00241", "00242", "00245", "00246", "00247", "00248", "00250", "00251", "00252", "00253", "00254", "00255", "00257", "00258", "00259", "00260", "00262", "00263", "00264", "00265", "00266", "00267", "00270", "00271", "00273", "00274", "00277", "00278", "00280", "00281", "00284", "00285", "00287", "00288", "00289", "00290", "00292", "00294", "00295", "00296", "00297", "00298", "00300", "00302", "00305", "00308", "00310", "00313", "00314", "00315", "00317", "00318", "00320", "00321", "00322", "00324", "00328", "00331", "00333", "00334", "00335", "00338", "00341", "00342", "00343", "00344", "00346", "00349", "00351", "00352", "00355", "00356", "00357", "00358", "00359", "00360", "00362", "00363", "00364", "00365", "00366", "00368", "00369", "00371", "00377", "00379", "00380", "00384", "00386", "00390", "00392", "00394", "00395", "00397", "00398", "00399", "00400", "00402", "00404", "00405", "00406", "00408", "00409", "00414", "00416", "00418", "00419", "00420", "00421", "00422", "00423", "00425", "00426", "00428", "00429", "00433", "00436", "00437", "00438", "00439", "00441", "00445", "00447", "00450", "00451", "00457", "00460", "00465", "00467", "00471", "00473", "00474", "00475", "00476", "00479", "00480", "00481", "00483", "00484", "00485", "00487", "00488", "00489", "00492", "00494", "00495", "00497", "00499"]},
    "longvideobench": {"sha256": "59e138d85b68b51a9509a7a4c73b0b1f0b2c0868f4ab23446db35b39c86f5dec", "qids": ["-Xg2-SUq6wo_0", "-Xg2-SUq6wo_1", "-if_Wg43vTk_0", "-kaF6SnSEo8_0", "0WEnmqVVbHo_0", "0WEnmqVVbHo_1", "0ln2qdCR5lA_0", "0mEN5Jf2hU0_0", "0mEN5Jf2hU0_1", "0rWA-p4p5IM_1", "14ot4DrXdds_0", "1R5uPaL0V-0_1", "1oL4ugMdDvQ_0", "1pxrIj9Xyps_1", "2F7d7aUCmUU_0", "2W2ZkYARds4_1", "2vVQo_GMA70_1", "2zZSMnGLGao_1", "3Qv2K_0GEE0_0", "3avWNHoEDAg_0", "3hyPwjkdHEA_1", "3t_Knk7FWT8_0", "3u__SZlBLC0_0", "49YMA0f1yhI_0", "4ouAf1ldH60_2", "58mCJVNMJvw_0", "5zbV24vyO44_1", "6d-EVupvWzU_1", "6moss7hGbvg_0", "7F9IrtSHmc0_1", "7TljSpTBS9c_0", "88LbBgZP1vQ_1", "8MkL3W6wU3g_2", "8Qe03WDCrB4_0", "8hhcFRoR0mw_1", "8wbNEJjBa0k_0", "9PD3ciudpIE_1", "9aWPbYosJUw_0", "9yQV0t5meo4_0", "@_eat_sleep_travel_repeat-7296468146473536800_0", "@healthfood-6874943613630041350_0", "@healthfood-6948509102305791238_0", "@healthfood-7115052132578921774_0", "@healthfood-7125042270381853995_0", "@healthfood-7135100078221430058_0", "@healthfood-7200404243168415022_0", "@jess.morg-7195624647830850858_0", "@jess.morg-7255843192677911851_0", "@jess.morg-7275475379375164715_0", "@jess.morg-7291800958152084782_0", "@jetset_anna-6933580627824495878_0", "@jetset_anna-7126584501228129542_0", "@jonijawne-7184469162956246277_0", "@jonijawne-7197144530024500485_0", "@jonijawne-7214559428471770374_0", "@jonijawne-7305901654417706246_0", "@jonijawne-7312063059160206597_0", "@kelseyinlondon-7094322812327906565_0", "@kerstinong-6900436106790194434_0", "@kerstinong-7197781091162279169_0", "@kerstinong-7254520467002854657_0", "@lisolna-7203500499360877829_0", "@lisolna-7235995901477604635_0", "@lisolna-7282789187676294432_0", "@lisolna-7288701821369978144_0", "@lisolna-7321010343067618592_0", "@lisolna-7359951777845775648_0", "@luxtravelbe-7233851486474636570_0", "@luxtravelbe-7286546546932370721_0", "@movie.explained6-7253164232307379457_0", "@movie.explained6-7253792552996867330_0", "@movie.explained6-7254803496900267266_0", "@movie.explained6-7257359603489443073_0", "@movie.explained6-7259227637992705282_0", "@movie.explained6-7262938043315686664_0", "@movie.explained6-7268936523481943297_0", "@movie.explained6-7270293967017725185_0", "@movie.explained6-7275653401025826050_0", "@movie.explained6-7277140989259599105_0", "@movie.explained6-7277548218106367234_0", "@movie.explained6-7278310085217094914_0", "@placesunleashed-7289528053112343814_0", "@placesunleashed-7297000148201262341_0", "@placesunleashed-7303594391850044678_0", "@placesunleashed-7308098412078025989_0", "@placesunleashed-7324498849857326341_0", "@placesunleashed-7337436458078260485_0", "@recipesbyanne-7141686631676808454_0", "@recipesbyanne-7153617149041446150_0", "@recipesbyanne-7161798161395240197_0", "@recipesbyanne-7231566545263103259_0", "@thatrecipe.us-7266555380925287723_0", "@thatrecipe.us-7301855783438896427_0", "@thatrecipe.us-7335998138500648235_0", "@tiffycooks-6940704678569053446_0", "@tiffycooks-7062861483687873798_0", "@tiffycooks-7080578381569445125_0", "@tiffycooks-7171182579058511110_0", "AHeq99pojLo_0", "Aau-XoIebno_2", "BtaVRhoLpC0_0", "CEZ9rbjK3P4_0", "DVsw1brd_Yc_0", "DX-mYMYf8jI_0", "Dkm35G5kkcc_0", "DoizYSYQRqU_2", "E7FSg22MdKE_1", "ErGYJ7kqIow_0", "F2OhCCEIOcU_0", "F2OhCCEIOcU_1", "F2i-LchFVCU_0", "F4bDyyEO4PU_0", "F4bDyyEO4PU_1", "F8Ma1qs0Rkg_1", "Fq3zbbp-lv4_1", "GAoK4XjssrM_0", "GHScTyR2OgQ_0", "GZFL58_pXPg_0", "GwcXYTX2GmA_0", "H2ksp6sRR-k_2", "H7gnsO4UZmU_0", "HAED3riiZkw_0", "HPunfsyjETs_0", "HWyDOQrYtCk_0", "I2Fyzav8MxE_0", "IN0osLg-Mn8_1", "Ip9DbdOtqF4_1", "J1IwKg2ufk8_2", "JIoE18JYGcM_0", "J_ZmaKRpyoU_2", "Jfp1Ks7Hh1E_1", "Jfp1Ks7Hh1E_2", "Jim_NG0PYpc_0", "JpjytHmGHZ4_0", "LAQRjgx_OY8_1", "LYvStKy8iAc_1", "Lc7RikDaa30_0", "LcZxjqtzXJI_1", "Lf7uW4xK2bc_0", "LfUsGv-ESbc_1", "LlqsCCa6y58_2", "M-YfPangEfA_0", "M5Umm-ltnao_0", "M7YSCIkUaNw_1", "MJYBHfYF8LI_1", "MTmUo6QITo4_0", "Msz128EJeWE_0", "MvgJAD6tZXo_2", "MyDAFlHsYsQ_0", "NHIT9vq6mJU_1", "Ng2rNm6Nwsg_0", "NpYUxd1vUUE_2", "O471uwTNx6k_2", "P0BSTjziVys_0", "PE32sjfN-uM_0", "PbiTIR8N4Hc_1", "PzUxuZ-KGsU_0", "QHS9ZZBdK-g_0", "QPth_xqBXGY_2", "RTUFPjliMCU_0", "SYqEMs0EYoI_0", "Sn7JPKbG6tY_2", "T15Kv6dtYO0_0", "T15Kv6dtYO0_2", "TMe7oXMJoSM_2", "TXiX3NO5f5w_0", "TlaX2iIYZD4_1", "TynnVVRNZGs_0", "UD5ifzOPzhc_0", "UN3ICsfqKEY_1", "UbNyMSwoT5A_0", "UbgwG8fcIu0_0", "UwJTCg5fpXg_1", "Uy_o-WCq2Cc_0", "V0h7rJShw0g_0", "VLInjyogciw_0", "VNvH5u5CWV8_0", "VPcdtoldWCA_0", "VkNF0rXuDXw_1", "Vn-z0qXp9bQ_0", "VwZeSoYugZk_1", "WHxBO7XecSY_0", "WaiGdRYD36k_1", "X5v4nBo5y28_2", "XJ6REZOXsvM_0", "XVXczyheik0_1", "Y1YCvEip_ko_2", "YcbKamVxDzI_1", "YcbKamVxDzI_2", "YehqA9xoGTY_1", "YiHjsOqxtk0_0", "Z6Hx_BAZeUw_0", "Z6Hx_BAZeUw_2", "Z7Cox6lPW3c_1", "ZoUsR8t8IxE_0", "ZoUsR8t8IxE_1", "_EUDpS9UF9o_0", "aVHAr8rc-Ks_1", "aX_HgA5SNLQ_0", "akoJDx23QWU_0", "anQ5KFW8Gn4_0", "aqUisZS9Ruw_0", "aqUisZS9Ruw_1", "bDpgz_2Piqg_0", "bH1Q9AM5vik_0", "baFsMWNavQ4_0", "d8H7hgQY9ew_0", "d8H7hgQY9ew_1", "dE5iWeCVpGI_0", "e-EtQHH4aWE_0", "e11Q4ThFu5A_0", "eE5Z7gDbgVA_1", "e_nAKiSutiA_0", "er1oRjH2iu8_1", "fO7nwCix8xU_0", "fPLjjr8w6DU_1", "fZBC3nmvJb8_0", "fryyNwUCPWA_1", "fvCrE5NCsts_1", "g7zuBUMBr2E_0", "gAgCnu82RHE_1", "gJuOMPiixUA_0", "gtX_oRpLClY_0", "gyV6EqgiPNg_0", "h0OHi9uAcBo_0", "h4jIoMxZopU_0", "h4jIoMxZopU_1", "hf4WUOagFAw_1", "hznvV2bBkX4_0", "i6C6r2g4Y7Q_0", "i8TJ7RgimNM_0", "iHNjWhx3EaI_0", "iJgh2dnudIU_1", "ie9NxdgNe0g_0", "iwXp1fT89-M_1", "iwXp1fT89-M_2", "ixAU3l0sX_o_0", "jJGbXCCU5yc_0", "jbPR2SJuFHg_0", "jvkmcX47bKU_1", "jvkmcX47bKU_2", "k194yU67Zik_0", "k4jiEuZbN-4_0", "kFHVBCEwC3w_0", "kWUmHAzCp7s_0", "kbRtl58u_kk_0", "l3vxOGgAM2g_0", "lN3WnXMaE0o_0", "lN3WnXMaE0o_1", "lNReCCShKJQ_0", "lcEaHk8f4Co_0", "luRqMb5qfhM_0", "m1mGerQhdHA_0", "m7sL8LuMD8Q_0", "mB7NW91REF4_0", "mFcEWmtn3ag_1", "mHARxee4EzQ_0", "mS1QPVgBDQo_1", "mS1QPVgBDQo_2", "mTn_C-SyW84_0", "mbcvVYobCXI_1", "mld0TnA2jEs_1", "nBWB-h68E4k_0", "nvG2YMWiwqU_0", "o2F-N42Ufo4_2", "oCXKARwr6PA_0", "oO8OJZFZT9s_0", "oddHY1vwcjo_0", "pFtKaT3GF9I_0", "pPJq1rMDRGs_1", "q3FAxTSENEw_1", "qZVBFAtfp2A_0", "qbA42wQoWAs_0", "qksR2Zvd-FM_2", "qsH6q5wNso4_1", "rP7sQe784k8_1", "rwL_XPw46zQ_1", "rzIiQ4Vxlbk_0", "t1nhAnMQBHg_0", "t1nhAnMQBHg_1", "t3P11ENBZyc_1", "tdA5atpqaAc_0", "u1D4ArcBjLI_1", "u5NAcHhI_Uc_0", "uDZrl35yeZE_0", "v-7zF3Y0yJs_0", "v0QLje6xYgA_0", "vJ9hYCUDHTo_0", "vJ9hYCUDHTo_1", "vVRC-0VKPrg_2", "vix2_R79-l4_0", "wGWbI3BHVM8_0", "wSHPuI7wWIg_2", "xSzXFQPldJg_0", "yFAuXmcGk2Y_0", "yqejTvYILlA_2", "ysRFFN5nzqE_2", "yz3lOAe32Tw_0", "z8YF-D-t7AI_0", "zVudr8cxHRE_1"]},
}
print({b: len(v["qids"]) for b, v in REP_MANIFEST.items()}, "questions per benchmark")

### 1 · Setup

In [ ]:
# ── Setup: folders, environment, small installs ──────────────────────
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_home")      # keep HF cache on the big temp disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "WARN ") + "pip install " + " ".join(pkgs))

pip_install("hf_xet")   # faster Hugging Face downloads (harmless if already present)
pip_install("av")       # backup video decoder

# Same library version as the main study, so every model runs on identical library code.
# (Checked before transformers is imported, so no restart is needed.)
from importlib.metadata import version as _v, PackageNotFoundError
TRANSFORMERS_PIN = "5.16.1"
try:
    _have = _v("transformers")
except PackageNotFoundError:
    _have = None
if _have != TRANSFORMERS_PIN:
    print(f"transformers {_have} found -> installing {TRANSFORMERS_PIN} (the version used in the main study)")
    pip_install(f"transformers=={TRANSFORMERS_PIN}")
    assert _v("transformers") == TRANSFORMERS_PIN, "transformers pin failed — send me this log"
print("transformers", _v("transformers"), "| torch", _v("torch"), "| python", sys.version.split()[0])

if MODEL == "qwen3vl":
    from transformers import Qwen3VLForConditionalGeneration  # noqa: F401
else:
    from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401
print("transformers OK")

### 1b · Run record (+ resume from an earlier session)

In [ ]:
# ── Run record + resume from an earlier session ──────────────────────
import json, datetime, glob, shutil
RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
ERR_PATH = os.path.join(OUT_DIR, "errors.log")
RUN_RECORD = {
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "benchmark": BENCHMARK, "model": MODEL_ID, "frames": NUM_FRAMES, "quick_test": QUICK_TEST, "study": "replication",
    "versions": {"transformers": _v("transformers"), "torch": _v("torch"), "python": sys.version.split()[0]},
    "preregistration": PREREG_URL, "sample_seed": SAMPLE_SEED,
    "videomme_chunks": VIDEOMME_CONFIRM_CHUNKS, "manifest_sha256": REP_MANIFEST[BENCHMARK]["sha256"],
    "methods": METHOD_GRID, "flashvid": FLASHVID,
    **({"holitom": {**HOLITOM, "T": {str(k): v for k, v in HOLITOM["T"].items()}}} if MODEL == "llava_video" else
       {"min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS}),
    "noise": {"frame_shifts": NOISE_FRAME_SHIFTS, "option_orders": NOISE_OPTION_ORDERS},
}
with open(os.path.join(OUT_DIR, f"run_record_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"), "w") as f:
    json.dump(RUN_RECORD, f, indent=2)

# If a previous session's results.jsonl for this benchmark is attached (Add Input), continue from it.
if not os.path.exists(RES_PATH):
    # matches e.g. .../flips_rep_qwen3vl_egoschema/results.jsonl or a dataset named "flips-rep-qwen3vl-egoschema"
    prev = [p for p in glob.glob("/kaggle/input/**/results.jsonl", recursive=True)
            if f"{MODEL}_{BENCHMARK}" in p.lower().replace("-", "_") and "quick" not in p.lower()]
    if prev:
        shutil.copy(prev[0], RES_PATH)
        print("Resuming from", prev[0])
print(json.dumps(RUN_RECORD, indent=1)[:1500])

### 2 · Imports and GPU check

In [ ]:
# ── Imports + hardware check ──────────────────────────────────────────
import json, math, time, random, shutil, zipfile, gc, heapq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
import transformers
from transformers import AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
assert N_GPU > 0 or DATA_CHECK_ONLY, "No GPU found! Right panel > Settings > Accelerator > 'GPU T4 x2', then Run All again."
for i in range(N_GPU):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk for temp videos: {free_gb:.1f} GB")
if free_gb < 12:
    print("⚠️ Less than 12 GB free — a 5 GB video chunk needs ~11 GB while unzipping.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Optional: a Hugging Face token stored as a Kaggle Secret named HF_TOKEN (Add-ons > Secrets).
# Not needed for these public repos, but harmless.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret (fine for public repos)")

### 3 · Benchmarks: tables, the replication sample, downloads, frames

In [ ]:
# ── Data: benchmark tables, pre-registered sampling, video download, frame reading ──
import io, tarfile, hashlib, re, concurrent.futures as cf
from huggingface_hub import list_repo_files

LETTERS = "ABCDEFGH"
_PREFIX = re.compile(r"^\s*\(?[A-Ha-h][\.\):]\s*")
_norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().rstrip(".").strip().lower()

def _opts(cands):
    """Plain candidate texts -> 'A. text' strings (any existing 'A.' / '(A)' prefix removed first)."""
    return [f"{LETTERS[i]}. {_PREFIX.sub('', str(c)).strip()}" for i, c in enumerate(cands)]

def _answer_letter(cands, answer):
    a = _norm(_PREFIX.sub("", str(answer)))
    hits = [i for i, c in enumerate(cands) if _norm(_PREFIX.sub("", str(c))) == a]
    return LETTERS[hits[0]] if len(hits) == 1 else None

def _hf(repo, fn, **kw):
    return hf_hub_download(repo, fn, repo_type="dataset", local_dir=os.path.join(WORK_DIR, repo.replace("/", "__")), **kw)

def _clean_cache():
    """Big archives leave a copy in the Hugging Face xet cache; remove it to keep disk free."""
    shutil.rmtree(os.path.join(os.environ.get("HF_HOME", ""), "xet"), ignore_errors=True)

def _sample(df, by, k, seed=SAMPLE_SEED):
    out = []
    for g, sub in df.groupby(by, sort=True):
        out.append(sub.sample(n=min(k, len(sub)), random_state=seed) if k else sub)
    return pd.concat(out).sort_values("qid", kind="stable").reset_index(drop=True)

# ═════ Video-MME (questions per chunk; videos come chunk by chunk) ═════
VMME_REPO = "lmms-eval/Video-MME"

def videomme_table():
    files = list_repo_files(VMME_REPO, repo_type="dataset")
    pq = [f for f in files if f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(VMME_REPO, f)) for f in pq], ignore_index=True)
    low = {c.lower(): c for c in df.columns}
    col = lambda *ns: next((low[n] for n in ns if n in low), None)
    out = pd.DataFrame({
        "video_key": df[col("videoid", "video_id")].astype(str),
        "video_alt": df[col("video_id", "videoid")].astype(str),
        "qid": df[col("question_id")].astype(str),
        "group": df[col("duration")].astype(str),
        "task": df[col("task_type")].astype(str),
        "question": df[col("question")].astype(str),
        "options": df[col("options")].apply(lambda o: [str(x) for x in list(o)]),
        "answer": df[col("answer")].astype(str).str.strip().str[0].str.upper(),
    })
    return out

VIDEO_EXT = (".mp4", ".mkv", ".webm", ".avi", ".mov")

def videomme_fetch_chunk(chunk):
    fn = f"videos_chunked_{chunk}.zip"
    t0 = time.time()
    zpath = _hf(VMME_REPO, fn)
    vdir = os.path.join(WORK_DIR, f"vmme_{chunk}")
    os.makedirs(vdir, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(VIDEO_EXT):
                with z.open(m) as src, open(os.path.join(vdir, os.path.basename(m.filename)), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove(zpath); _clean_cache()
    vids = {os.path.splitext(f)[0]: os.path.join(vdir, f) for f in os.listdir(vdir)}
    print(f"Video-MME chunk {chunk}: {len(vids)} videos in {time.time() - t0:.0f}s")
    return vids

# ═════ MVBench (video branch: pre-cut segments, one file per video) ═════
MVB_REPO, MVB_REV = "OpenGVLab/MVBench", "video"
MVB_FOLDERS = {   # lmms-eval DATA_LIST
    "object_interaction": "star/Charades_segment", "action_sequence": "star/Charades_segment",
    "action_prediction": "star/Charades_segment", "action_localization": "sta/sta_video_segment",
    "moving_count": "clevrer/video_validation", "character_order": "perception/videos",
    "object_shuffle": "perception/videos", "egocentric_navigation": "vlnqa",
    "moving_direction": "clevrer/video_validation", "episodic_reasoning": "tvqa/video_fps3_hq_segment",
    "fine_grained_action": "Moments_in_Time_Raw/videos", "scene_transition": "scene_qa/video",
    "state_change": "perception/videos", "moving_attribute": "clevrer/video_validation",
    "action_antonym": "ssv2_video_mp4", "unexpected_action": "FunQA_test/test",
    "counterfactual_inference": "clevrer/video_validation", "object_existence": "clevrer/video_validation",
    "action_count": "perception/videos",
}   # fine_grained_pose (NTU RGB+D) is excluded by the pre-registration

def _read_json_any(path):
    txt = open(path, encoding="utf-8").read().strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(l) for l in txt.splitlines() if l.strip()]

MVB_FILES = None
def mvbench_table():
    global MVB_FILES
    MVB_FILES = list_repo_files(MVB_REPO, repo_type="dataset", revision=MVB_REV)
    rows, dropped = [], 0
    for task, folder in MVB_FOLDERS.items():
        jf = next((f for f in MVB_FILES if f.endswith(f"json/{task}.json")), f"json/{task}.json")
        for i, d in enumerate(_read_json_any(_hf(MVB_REPO, jf, revision=MVB_REV))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{folder}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MVBench: {len(rows)} questions in {len(MVB_FOLDERS)} tasks ({dropped} dropped: answer not among candidates)")
    return pd.DataFrame(rows)

def mvbench_fetch(keys):
    all_files, by_base = set(MVB_FILES), {}
    for f in MVB_FILES:
        by_base.setdefault(os.path.basename(f), []).append(f)
    def one(k):
        cand = [k] if k in all_files else [f for f in by_base.get(os.path.basename(k), [])
                                         if f.startswith(k.split("/")[0])] or by_base.get(os.path.basename(k), [])
        if not cand:
            raise FileNotFoundError(k)
        return k, _hf(MVB_REPO, cand[0], revision=MVB_REV)
    return _parallel_fetch(keys, one)

# ═════ EgoSchema (public 500-question subset) ═════
EGO_REPO = "lmms-lab/egoschema"
def egoschema_table():
    files = list_repo_files(EGO_REPO, repo_type="dataset")
    pq = [f for f in files if f.startswith("Subset/") and f.endswith(".parquet")]
    df = pd.concat([pd.read_parquet(_hf(EGO_REPO, f)) for f in pq], ignore_index=True)
    rows = []
    for _, d in df.iterrows():
        opts = [str(o) for o in list(d["option"])]
        rows.append({"qid": str(d["question_idx"]), "video_key": str(d["video_idx"]), "video_alt": "",
                     "group": "ego", "task": "ego", "question": str(d["question"]), "options": _opts(opts),
                     "answer": LETTERS[int(str(d["answer"]).strip())]})
    print(f"EgoSchema subset: {len(rows)} questions")
    return pd.DataFrame(rows)

def egoschema_fetch(keys):
    need, got = set(keys), {}
    files = sorted(f for f in list_repo_files(EGO_REPO, repo_type="dataset") if f.endswith(".zip"))
    vdir = os.path.join(WORK_DIR, "ego_videos"); os.makedirs(vdir, exist_ok=True)
    for fn in files:
        if not need - set(got):
            break
        t0 = time.time()
        zpath = _hf(EGO_REPO, fn)
        with zipfile.ZipFile(zpath) as z:
            for m in z.infolist():
                stem = os.path.splitext(os.path.basename(m.filename))[0]
                if stem in need and stem not in got and m.filename.lower().endswith(VIDEO_EXT):
                    dst = os.path.join(vdir, os.path.basename(m.filename))
                    with z.open(m) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out, 1 << 24)
                    got[stem] = dst
        os.remove(zpath); _clean_cache()
        print(f"  {fn}: {len(got)}/{len(need)} videos found ({time.time() - t0:.0f}s, "
              f"free disk {shutil.disk_usage(WORK_DIR).free / 1e9:.0f} GB)")
    return got

# ═════ LongVideoBench (validation, video only) ═════
LVB_REPO = "longvideobench/LongVideoBench"
def longvideobench_table():
    files = list_repo_files(LVB_REPO, repo_type="dataset")
    if "lvb_val.json" in files:
        data = _read_json_any(_hf(LVB_REPO, "lvb_val.json"))
    else:
        data = pd.read_parquet(_hf(LVB_REPO, next(f for f in files if f.startswith("validation")))).to_dict("records")
    rows = []
    for d in data:
        cands = d.get("candidates")
        if cands is None:
            cands = [d[f"option{i}"] for i in range(5) if d.get(f"option{i}") not in (None, "N/A")]
        cands = [str(c) for c in list(cands)]
        rows.append({"qid": str(d["id"]), "video_key": os.path.basename(str(d["video_path"])), "video_alt": "",
                     "group": str(d["duration_group"]), "task": str(d.get("question_category", "")),
                     "question": str(d["question"]), "options": _opts(cands),
                     "answer": LETTERS[int(d["correct_choice"])]})
    print(f"LongVideoBench val: {len(rows)} questions")
    return pd.DataFrame(rows)

class _PartsReader(io.RawIOBase):
    """Reads videos.tar.part.aa, .ab, ... as one stream; downloads each part when needed, deletes it after."""
    def __init__(self, parts):
        self.parts, self.k, self.f, self.path = list(parts), -1, None, None
    def readable(self):
        return True
    def _next(self):
        if self.f:
            self.f.close(); os.remove(self.path); _clean_cache()
        self.k += 1
        if self.k >= len(self.parts):
            self.f = None
            return False
        t0 = time.time()
        self.path = _hf(LVB_REPO, self.parts[self.k])
        self.f = open(self.path, "rb")
        print(f"  {self.parts[self.k]} ready ({time.time() - t0:.0f}s)")
        return True
    def readinto(self, b):
        while True:
            if self.f is None and not self._next():
                return 0
            n = self.f.readinto(b)
            if n:
                return n
            if not self._next():
                return 0

def longvideobench_fetch(keys):
    need, got = set(keys), {}
    parts = sorted(f for f in list_repo_files(LVB_REPO, repo_type="dataset") if f.startswith("videos.tar.part"))
    vdir = os.path.join(WORK_DIR, "lvb_videos"); os.makedirs(vdir, exist_ok=True)
    reader = io.BufferedReader(_PartsReader(parts), buffer_size=1 << 24)
    with tarfile.open(fileobj=reader, mode="r|") as tar:
        for m in tar:
            b = os.path.basename(m.name)
            if m.isfile() and b in need and b not in got:
                with tar.extractfile(m) as src, open(os.path.join(vdir, b), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
                got[b] = os.path.join(vdir, b)
            if not need - set(got):
                break
    return got

# ═════ MLVU (dev, the 7 multiple-choice tasks; one file per video) ═════
MLVU_REPO = "MLVU/MVLU"
MLVU_TASKS = ["1_plotQA", "2_needle", "3_ego", "4_count", "5_order", "6_anomaly_reco", "7_topic_reasoning"]
MLVU_FILES = None
def mlvu_table():
    global MLVU_FILES
    MLVU_FILES = list_repo_files(MLVU_REPO, repo_type="dataset")
    rows, dropped = [], 0
    for task in MLVU_TASKS:
        jf = next(f for f in MLVU_FILES if f.endswith(f"json/{task}.json"))
        for i, d in enumerate(_read_json_any(_hf(MLVU_REPO, jf))):
            ans = _answer_letter(d["candidates"], d["answer"])
            if ans is None or len(d["candidates"]) > len(LETTERS):
                dropped += 1
                continue
            rows.append({"qid": f"{task}-{i:04d}", "video_key": f"{task}/{d['video']}", "video_alt": "",
                         "group": task, "task": task, "question": str(d["question"]),
                         "options": _opts(d["candidates"]), "answer": ans})
    print(f"MLVU dev MC: {len(rows)} questions ({dropped} dropped)")
    return pd.DataFrame(rows)

def mlvu_fetch(keys):
    by_rel = {"/".join(f.split("/")[-2:]): f for f in MLVU_FILES if f.lower().endswith(VIDEO_EXT)}
    def one(k):
        if k not in by_rel:
            raise FileNotFoundError(k)
        return k, _hf(MLVU_REPO, by_rel[k])
    return _parallel_fetch(keys, one)

def _parallel_fetch(keys, one, workers=8):
    got, t0 = {}, time.time()
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(one, k): k for k in sorted(set(keys))}
        for i, fu in enumerate(cf.as_completed(futs)):
            try:
                k, p = fu.result()
                got[k] = p
            except Exception as e:
                with open(ERR_PATH, "a") as f:
                    f.write(f"{futs[fu]}\tdownload\t{type(e).__name__}: {e}\n")
            if (i + 1) % 100 == 0:
                print(f"  downloaded {i + 1}/{len(futs)} ({time.time() - t0:.0f}s)")
    print(f"Downloaded {len(got)}/{len(futs)} videos in {time.time() - t0:.0f}s")
    return got

# ═════ Manifest: the pre-registered sample, frozen to a file with its hash ═════
def _valid(df):
    """Pre-registration: questions with fewer than two options carry no information and are removed before sampling."""
    bad = df.options.apply(len) < 2
    if bad.any():
        print(f"Removed {int(bad.sum())} question(s) with fewer than two options: {df.qid[bad].tolist()}")
    return df[~bad]

def build_manifest():
    """The replication sample: the frozen question IDs in REP_MANIFEST (a subset of the main study's manifest)."""
    want = REP_MANIFEST[BENCHMARK]
    ids = sorted(want["qids"])
    assert hashlib.sha256("\n".join(ids).encode()).hexdigest() == want["sha256"], "replication manifest corrupted"
    full = {"videomme": videomme_table, "egoschema": egoschema_table,
            "longvideobench": longvideobench_table}[BENCHMARK]()
    df = _valid(full)
    df = df[df.qid.isin(set(ids))].sort_values("qid", kind="stable").reset_index(drop=True)
    missing = sorted(set(ids) - set(df.qid))
    if missing:
        raise RuntimeError(f"{len(missing)} replication questions not found in the benchmark table, e.g. {missing[:3]}")
    with open(os.path.join(OUT_DIR, f"manifest_{BENCHMARK}.json"), "w") as f:
        json.dump({"benchmark": BENCHMARK, "seed": SAMPLE_SEED, "n": len(ids), "sha256": want["sha256"],
                   "qids": ids}, f)
    print(f"Replication manifest: {len(df)} questions, {df.video_key.nunique()} videos, sha256 {want['sha256'][:16]}…")
    print("Per group:", df.group.value_counts().sort_index().to_dict())
    return df

# ═════ Frames ═════
def _frame_indices(total, n, shift):
    """Uniform sampling as in LMMs-Eval (np.linspace over all frames). shift = fraction of one sampling
    interval to move every sample later (noise passes)."""
    idx = np.linspace(0, total - 1, n)
    if shift:
        idx = idx + shift * (total - 1) / max(n - 1, 1)
    return np.clip(idx, 0, total - 1).round().astype(int)

def _read_cv2(path, n, shift=0.0, meta=None):
    import cv2
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0) or None
    frames, used = [], []
    if total > 0:
        for k in _frame_indices(total, n, shift):
            ok = False
            for off in (0, -1, 1, -5, 5, -20):
                kk = int(np.clip(k + off, 0, total - 1))
                cap.set(cv2.CAP_PROP_POS_FRAMES, kk)
                ok, fr = cap.read()
                if ok:
                    break
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); used.append(kk)
    cap.release()
    if meta is not None:
        meta.update(fps=fps, total=total, indices=used)
    return frames

def _read_av(path, n, shift=0.0, meta=None):
    import av
    out, used = [], []
    with av.open(path) as c:
        s = c.streams.video[0]
        dur = float(s.duration * s.time_base) if s.duration else float(c.duration or 0) / 1e6
        fps = float(s.average_rate) if s.average_rate else None
        end = max(dur - 0.5, 0)
        ts = np.linspace(0, end, n)
        if shift:
            ts = np.clip(ts + shift * end / max(n - 1, 1), 0, end)
        for t in ts:
            c.seek(int(t / s.time_base), stream=s, any_frame=False, backward=True)
            for fr in c.decode(s):
                out.append(fr.to_ndarray(format="rgb24")); used.append(int(round(t * (fps or 24))))
                break
    if meta is not None:
        meta.update(fps=fps, total=int(round(dur * (fps or 24))), indices=used)
    return out

def read_frames(path, n, shift=0.0, meta=None):
    """n frames sampled uniformly; meta (optional dict) receives fps, total frame count and the frame indices used."""
    frames = []
    try:
        frames = _read_cv2(path, n, shift, meta)
    except Exception as e:
        print("cv2 failed:", e)
    if len(frames) < n:
        try:
            frames = _read_av(path, n, shift, meta)
        except Exception as e:
            print("av failed:", e)
    if len(frames) == 0:
        raise RuntimeError(f"Could not decode {path}")
    if meta is not None:
        idx = list(meta.get("indices", []))[:len(frames)]
        while len(idx) < n:
            idx.append(idx[-1] if idx else 0)
        meta["indices"] = idx[:n]
    while len(frames) < n:
        frames.append(frames[-1])
    return frames[:n]

def read_all_frame_sets(path):
    """The three frame sets every video needs: original, and the two shifted noise passes (with their metadata)."""
    out, metas = {}, {}
    for k, s in [("orig", 0.0)] + list(NOISE_FRAME_SHIFTS.items()):
        metas[k] = {}
        out[k] = read_frames(path, NUM_FRAMES, shift=s, meta=metas[k])
    out["_meta"] = metas
    return out

# ═════ Option-order noise passes ═════
def _new_index(kind, j, n):
    """Which ORIGINAL option is shown at new position j."""
    return (j + 1) % n if kind == "option_cycle" else n - 1 - j

def reorder_options(options, kind):
    body = [_PREFIX.sub("", str(o)).strip() for o in options]
    n = len(body)
    return [f"{LETTERS[j]}. {body[_new_index(kind, j, n)]}" for j in range(n)]

def map_back(res, n, kind):
    """Express a reordered result in the original option letters."""
    lp_new = np.array(res["logp"])
    lp = np.empty(n)
    for j in range(n):
        lp[_new_index(kind, j, n)] = lp_new[j]
    srt = np.sort(lp)[::-1]
    out = dict(res)
    out.update(pred=LETTERS[int(lp.argmax())], logp=lp.round(4).tolist(), margin=float(srt[0] - srt[1]))
    return out

### 4 · Model, T4 memory fix, our decoder loop, prefix caching

In [ ]:
# ── Model: Qwen3-VL-8B-Instruct with direct control over visual tokens ─────
# ── T4 memory fix + safe attention routing ─────────────────────────────
# 1) Expand grouped K/V heads ourselves so PyTorch can use the memory-efficient kernel (math kernel -> OOM on T4).
# 2) Never hand an explicit mask to the memory-efficient kernel: on T4 that path can crash with
#    "illegal memory access". A plain causal mask is converted to is_causal=True (identical maths).
# 3) Guard: if any NON-causal mask (e.g. transformers mistaking position gaps for packed sequences)
#    reaches a long attention call, stop loudly instead of computing something silently wrong.
from torch.nn.attention import sdpa_kernel, SDPBackend
if not hasattr(torch.nn.functional, "_astra_orig_sdpa"):
    torch.nn.functional._astra_orig_sdpa = torch.nn.functional.scaled_dot_product_attention
_orig_sdpa = torch.nn.functional._astra_orig_sdpa
_TRIL_CACHE = {}
def _plain_causal(mask, L, S):
    if L != S or mask.dim() != 4 or mask.shape[0] != 1 or mask.shape[1] != 1:
        return False
    key = (L, str(mask.device))
    if key not in _TRIL_CACHE:
        _TRIL_CACHE.clear()
        _TRIL_CACHE[key] = torch.ones(L, L, dtype=torch.bool, device=mask.device).tril()
    allowed = mask[0, 0] if mask.dtype == torch.bool else (mask[0, 0] == 0)
    return torch.equal(allowed, _TRIL_CACHE[key])

def _sdpa_safe(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False, **kw):
    if enable_gqa and key.shape[-3] != query.shape[-3]:
        rep = query.shape[-3] // key.shape[-3]
        key = key.repeat_interleave(rep, dim=-3)
        value = value.repeat_interleave(rep, dim=-3)
    if attn_mask is not None:
        L, S = query.shape[-2], key.shape[-2]
        if _plain_causal(attn_mask, L, S):
            attn_mask, is_causal = None, True
        elif L > 1024:
            raise RuntimeError("A non-causal attention mask reached a long attention call — stopping instead of "
                               "computing silently wrong results. Send me this error.")
        elif query.dtype == torch.float16:
            # short question block over a cached prefix: compute in fp32 (cheap at this size), so cached passes
            # match the fp32-accumulating kernel used for full passes as closely as possible
            m = attn_mask if attn_mask.dtype == torch.bool else attn_mask.float()
            return _orig_sdpa(query.float(), key.float(), value.float(), attn_mask=m, dropout_p=dropout_p,
                              is_causal=is_causal, scale=scale, **kw).to(query.dtype)
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

torch.nn.functional.scaled_dot_product_attention = _sdpa_safe
NO_MATH = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION]
print("T4 attention fix applied (K/V heads expanded, causal masks routed to is_causal, math kernel disabled for the LLM)")


# Qwen3-VL specifics handled here: (1) 2 frames form one temporal patch, so 32 frames give 16 temporal groups, each
# introduced by a timestamp "<t seconds>" and wrapped in vision-start/end tokens; (2) 3-D (time, height, width) rotary
# positions; (3) DeepStack: features from three vision layers are added to the visual positions after LLM layers 0-2.
# Compression keeps each kept token's own 3-D position and its own DeepStack features (as FlashVID's official code).
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb, apply_rotary_pos_emb_vision, repeat_kv
from transformers.video_utils import VideoMetadata
import torchvision.transforms.functional as TVF

max_memory = {i: "13GiB" for i in range(N_GPU)}
if N_GPU >= 2:
    max_memory[0] = "9GiB"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", max_memory=max_memory, attn_implementation="sdpa", dtype=torch.float16).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer

def _get(root, paths):
    for p in paths:
        obj, ok = root, True
        for part in p.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError(f"None of {paths} found — send me the error and transformers version")

visual  = _get(model, ["model.visual", "visual"])
body    = _get(model, ["model.language_model", "language_model"])
lm_head = _get(model, ["lm_head"])
CFG, TCFG, VCFG = model.config, model.config.text_config, model.config.vision_config
VIDEO_TOKEN_ID = CFG.video_token_id
VSTART_ID, VEND_ID = CFG.vision_start_token_id, CFG.vision_end_token_id
MERGE = VCFG.spatial_merge_size
FACTOR = VCFG.patch_size * MERGE                         # 32 pixels per merged token side
NLAYERS, HID = TCFG.num_hidden_layers, TCFG.hidden_size
NHEADS, NKV = TCFG.num_attention_heads, TCFG.num_key_value_heads
HEAD_DIM = getattr(TCFG, "head_dim", None) or HID // NHEADS
INTER = TCFG.intermediate_size
EMB_DEVICE = body.embed_tokens.weight.device
LM_DEVICE = lm_head.weight.device
VDEV = next(visual.parameters()).device
N_DEEPSTACK = len(VCFG.deepstack_visual_indexes)
print(f"LLM layers={NLAYERS} hidden={HID} heads={NHEADS}/{NKV} head_dim={HEAD_DIM} | DeepStack layers {N_DEEPSTACK} | "
      f"{FACTOR}px per visual token, 2 frames per temporal group")

LETTERS = "ABCDEFGH"
LETTER_IDS = []
for L in LETTERS:
    ids = tok.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"Letter {L} is not a single token"
    LETTER_IDS.append(ids[0])

PREFIX = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"

def _letter_list(n):
    L = list(LETTERS[:n])
    return L[0] if n == 1 else (f"{L[0]} or {L[1]}" if n == 2 else ", ".join(L[:-1]) + ", or " + L[-1])

def suffix_text(question, options):
    opts = "\n".join(str(o) for o in options)
    return ("\nSelect the best answer to the following multiple-choice question based on the video. "
            f"Respond with only the letter ({_letter_list(len(options))}) of the correct option.\n"
            f"{question}\n{opts}\nThe best answer is:<|im_end|>\n<|im_start|>assistant\n")

def build_suffix_ids(question, options):
    """The main study's prompt (identical text)."""
    return tok(suffix_text(question, options), add_special_tokens=False, return_tensors="pt").input_ids[0]

def smart_resize_frame(h, w, factor=None, min_pixels=None, max_pixels=None):
    """qwen-vl-utils smart_resize (per frame), with FlashVID's baseline pixel limits."""
    factor = factor or FACTOR
    min_pixels, max_pixels = min_pixels or MIN_PIXELS, max_pixels or MAX_PIXELS
    h_bar = max(factor, round(h / factor) * factor)
    w_bar = max(factor, round(w / factor) * factor)
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((h * w) / max_pixels)
        h_bar = max(factor, math.floor(h / beta / factor) * factor)
        w_bar = max(factor, math.floor(w / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar = math.ceil(h * beta / factor) * factor
        w_bar = math.ceil(w * beta / factor) * factor
    return h_bar, w_bar

def prepare_video(frames, meta, prompt_tail=""):
    """Resize (bicubic, antialiased, as qwen-vl-utils) and run the official processor (patchify, normalise,
    timestamps). Returns the processor outputs for the text PREFIX + video (+ prompt_tail)."""
    h, w = frames[0].shape[:2]
    H, W = smart_resize_frame(h, w)
    v = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2)
    v = TVF.resize(v, [H, W], interpolation=TVF.InterpolationMode.BICUBIC, antialias=True)
    v = v.round().clamp(0, 255).to(torch.uint8).permute(0, 2, 3, 1).numpy()
    fps = (meta or {}).get("fps") or 24.0
    idx = list((meta or {}).get("indices") or range(len(frames)))
    md = VideoMetadata(total_num_frames=int((meta or {}).get("total") or (max(idx) + 1)), fps=float(fps),
                       frames_indices=idx, width=W, height=H)
    return processor(text=[PREFIX + processor.video_token + prompt_tail], videos=[v], video_metadata=[md],
                     do_resize=False, do_sample_frames=False, cap_pixels_per_frame=False, return_tensors="pt")

# ── FlashVID needs the last vision block's attention (mean over heads and queries, per temporal group) ──
_LAST_ATTN = {}
def _capture_last_attn(mod, args, kwargs):
    hs = args[0] if args else kwargs["hidden_states"]
    _LAST_ATTN["h"] = hs
    _LAST_ATTN["pe"] = kwargs.get("position_embeddings")
    _LAST_ATTN["cu"] = kwargs.get("cu_seqlens")
_hook = visual.blocks[-1].attn.register_forward_pre_hook(_capture_last_attn, with_kwargs=True)

@torch.no_grad()
def _flashvid_cls_attention(attn_mod, n_groups):
    """FlashVID Qwen3VLVisionAttention_forward(return_logits=True) + the pooling in its vision forward."""
    hs, (cos, sin) = _LAST_ATTN["h"], _LAST_ATTN["pe"]
    seq = hs.shape[0]
    q, k, _ = attn_mod.qkv(hs).reshape(seq, 3, attn_mod.num_heads, -1).permute(1, 0, 2, 3).unbind(0)
    q, k = apply_rotary_pos_emb_vision(q, k, cos, sin)
    q = q.transpose(0, 1).reshape(attn_mod.num_heads, n_groups, -1, attn_mod.head_dim).permute(1, 0, 2, 3)
    k = k.transpose(0, 1).reshape(attn_mod.num_heads, n_groups, -1, attn_mod.head_dim).permute(1, 0, 2, 3)
    out = []
    for i in range(n_groups):
        a = torch.matmul(q[i], k[i].transpose(-1, -2)) / attn_mod.head_dim ** 0.5
        a = torch.softmax(a, dim=-1, dtype=torch.float32).to(q.dtype)
        out.append(a.mean(0).mean(0))
    w = torch.stack(out, 0)                                   # (groups, patches per group)
    return w.view(n_groups, w.shape[-1] // (MERGE * MERGE), -1).mean(-1)

@torch.no_grad()
def encode_frames(frames, meta=None):
    """One frame set -> everything the passes need: the prefix token ids (with timestamps), their 3-D positions,
    the merged visual tokens (N, D), DeepStack features (3 x (N, D)), FlashVID's per-group attention (G, P)."""
    inp = prepare_video(frames, meta)
    ids = inp["input_ids"][0]
    grid = inp["video_grid_thw"]
    G, Hp, Wp = [int(x) for x in grid[0]]
    out = visual(inp["pixel_values_videos"].to(VDEV, torch.float16), grid_thw=grid.to(VDEV), return_dict=True)
    X = out.pooler_output if hasattr(out, "pooler_output") else out[0]
    DS = list(out.deepstack_features)
    CA = _flashvid_cls_attention(visual.blocks[-1].attn, G)
    _LAST_ATTN.clear()
    pos, _ = model.model.get_rope_index(ids[None], inp["mm_token_type_ids"], video_grid_thw=grid)
    pos = pos[:, 0]                                           # (3, L)
    vmask = ids == VIDEO_TOKEN_ID
    P = (Hp // MERGE) * (Wp // MERGE)
    assert int(vmask.sum()) == X.shape[0] == G * P, "visual token count mismatch"
    text_emb = body.embed_tokens(ids.to(EMB_DEVICE))
    return {"ids": ids, "pos": pos, "vmask": vmask, "X": X.to(EMB_DEVICE), "DS": [d.to(EMB_DEVICE) for d in DS],
            "CA": CA.to(EMB_DEVICE).float(), "G": G, "Hm": Hp // MERGE, "Wm": Wp // MERGE, "P": P,
            "text_emb": text_emb, "next_pos": int(pos.max()) + 1, "n_full": int(X.shape[0]),
            "pixel_values_videos": inp["pixel_values_videos"], "grid": grid}

def assemble(enc, groups, drop_start_end=False):
    """Builds a prefix from per-group visual tokens. groups[g] = (feat (k, D), [ds (k, D)] * 3, pos (3, k)).
    Text tokens (prompt, timestamps, vision start/end) keep their own embeddings and positions."""
    ids, pos, vm = enc["ids"], enc["pos"], enc["vmask"]
    dev = enc["text_emb"].device
    keep_text = ~vm
    if drop_start_end:
        keep_text &= (ids != VSTART_ID) & (ids != VEND_ID)
    vid_idx = torch.nonzero(vm).flatten()
    starts = vid_idx[::enc["P"]].tolist()                    # first index of each temporal group
    emb, pp, vmask, ds = [], [], [], [[] for _ in range(N_DEEPSTACK)]
    cur = 0
    for g in range(enc["G"]):
        s = starts[g]
        t = torch.arange(cur, s)
        t = t[keep_text[cur:s]]
        emb.append(enc["text_emb"][t.to(dev)]); pp.append(pos[:, t]); vmask.append(torch.zeros(len(t), dtype=torch.bool))
        f, dsl, gp = groups[g]
        emb.append(f.to(dev, enc["text_emb"].dtype)); pp.append(gp.cpu()); vmask.append(torch.ones(f.shape[0], dtype=torch.bool))
        for j in range(N_DEEPSTACK):
            ds[j].append(dsl[j])
        cur = s + enc["P"]
    t = torch.arange(cur, len(ids))
    t = t[keep_text[cur:]]
    emb.append(enc["text_emb"][t.to(dev)]); pp.append(pos[:, t]); vmask.append(torch.zeros(len(t), dtype=torch.bool))
    return {"embeds": torch.cat(emb, 0), "pos": torch.cat(pp, 1), "vmask": torch.cat(vmask, 0),
            "ds": [torch.cat(d, 0) for d in ds], "next_pos": enc["next_pos"]}

def _group_pos(enc):
    return enc["pos"][:, enc["vmask"]].view(3, enc["G"], enc["P"])

def full_spec(enc):
    G, P = enc["G"], enc["P"]
    gp = _group_pos(enc)
    X, DS = enc["X"].view(G, P, -1), [d.view(G, P, -1) for d in enc["DS"]]
    s = assemble(enc, [(X[g], [d[g] for d in DS], gp[:, g]) for g in range(G)])
    s.update(inner=None, nvis=enc["n_full"])
    return s

def llm_flops_layers(ns):
    """Prefill FLOPs of the LLM when layer l sees ns[l] tokens (q/o projections use heads x head_dim)."""
    dkv, dq = NKV * HEAD_DIM, NHEADS * HEAD_DIM
    return float(sum(2 * (2 * n * HID * dq + 2 * n * HID * dkv + 2 * n * n * dq + 3 * n * HID * INTER) for n in ns))

def llm_flops(n):
    return llm_flops_layers([n] * NLAYERS)

def _sync():
    for i in range(N_GPU):
        torch.cuda.synchronize(i)

@torch.no_grad()
def last_query_attention(layer, h, pe):
    """Attention of the last token over all tokens in this layer (FlashVID's output_attentions fallback)."""
    a = layer.self_attn
    dev = a.q_proj.weight.device
    h = h.to(dev)
    cos, sin = pe[0].to(dev), pe[1].to(dev)
    hn = layer.input_layernorm(h)
    q = a.q_norm(a.q_proj(hn[:, -1:]).view(1, 1, -1, HEAD_DIM)).transpose(1, 2)
    k = a.k_norm(a.k_proj(hn).view(1, hn.shape[1], -1, HEAD_DIM)).transpose(1, 2)
    q, _ = apply_rotary_pos_emb(q, q, cos[:, -1:], sin[:, -1:])
    _, k = apply_rotary_pos_emb(k, k, cos, sin)
    k = repeat_kv(k, NHEADS // NKV)
    w = torch.matmul(q, k.transpose(2, 3)) / HEAD_DIM ** 0.5
    return torch.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)

@torch.no_grad()
def apply_inner(h, pe, attn, inner, vmask):
    """FlashVID utils.fastv_prune with visual_pos_masks (its Qwen3-VL path)."""
    dev = h.device
    vm = vmask.to(dev)
    n_vis = int(vm.sum())
    n_keep = math.ceil(n_vis * inner["keep_ratio"])
    a = attn.mean(dim=1)[0, -1].to(dev)[vm]
    vis_idx = torch.nonzero(vm).flatten()
    top = torch.topk(a, k=n_keep).indices.sort().values
    keep = torch.sort(torch.cat([torch.nonzero(~vm).flatten(), vis_idx[top]])).values
    pe = (pe[0][..., keep.to(pe[0].device), :], pe[1][..., keep.to(pe[1].device), :])
    return h[:, keep], pe, n_keep

@torch.no_grad()
def forward_layers(x, pos3, vmask, ds, inner=None, nvis0=0):
    """Our own decoder loop (same calls as Qwen3VLTextModel.forward, incl. DeepStack); lets FlashVID prune inside."""
    pe = body.rotary_emb(x, pos3)
    h, attn, ns, nvis = x, None, [], []
    vis_now = nvis0
    vm_dev = vmask.to(x.device)
    for li, layer in enumerate(body.layers):
        if inner is not None and li == inner["layer"]:
            h, pe, vis_now = apply_inner(h, pe, attn, inner, vm_dev)
            attn = None
        if inner is not None and li == inner["layer"] - 1:
            attn = last_query_attention(layer, h, pe)
        ns.append(h.shape[1]); nvis.append(vis_now)
        h = layer(h, position_embeddings=pe, attention_mask=None, position_ids=None, use_cache=False)
        if isinstance(h, tuple):
            h = h[0]
        if ds is not None and li < len(ds):
            hd = h.device
            h = h.clone()
            h[0, vm_dev.to(hd)] = h[0, vm_dev.to(hd)] + ds[li].to(hd, h.dtype)
    h = body.norm(h.to(body.norm.weight.device))
    return h, ns, nvis

def _result(logits, n_opt, extra):
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]), **extra}

def _seq(spec, suf_ids=None):
    E = body.embed_tokens
    x, pos, vm = spec["embeds"], spec["pos"], spec["vmask"]
    if suf_ids is not None:
        suf = E(suf_ids.to(EMB_DEVICE))
        n_q = suf.shape[0]
        x = torch.cat([x, suf.to(x.dtype)], 0)
        pos = torch.cat([pos, (spec["next_pos"] + torch.arange(n_q)).view(1, -1).expand(3, -1)], 1)
        vm = torch.cat([vm, torch.zeros(n_q, dtype=torch.bool)], 0)
    return x[None].to(EMB_DEVICE), pos[:, None, :].to(EMB_DEVICE), vm

@torch.no_grad()
def llm_pass(spec, suf_ids, n_opt, manual=False):
    x, pos3, vm = _seq(spec, suf_ids)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    _sync(); t0 = time.time()
    inner = spec.get("inner")
    with sdpa_kernel(NO_MATH):
        if inner is None and not manual:
            out = body(inputs_embeds=x, position_ids=pos3, visual_pos_masks=vm[None].to(EMB_DEVICE),
                       deepstack_visual_embeds=spec["ds"], use_cache=False)
            h = out[0][0, -1]
            ns, nvis = [x.shape[1]] * NLAYERS, [spec["nvis"]] * NLAYERS
        else:
            hh, ns, nvis = forward_layers(x, pos3, vm, spec["ds"], inner, nvis0=spec["nvis"])
            h = hh[0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    return _result(logits, n_opt, {"ntok": int(spec["nvis"]), "vis_mean_layers": float(np.mean(nvis)), "sec": float(dt),
                                   "flops": llm_flops_layers(ns),
                                   "peak_gb": round(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9, 3)})

# ── Prefix caching ──
from transformers import DynamicCache

def _new_cache():
    try:
        return DynamicCache(config=TCFG)
    except TypeError:
        return DynamicCache()

def _crop(cache, n):
    extra = cache.get_seq_length() - n
    if extra > 0 and hasattr(cache, "crop"):
        cache.crop(-extra)
    if cache.get_seq_length() != n:
        for l in cache.layers:
            l.keys, l.values = l.keys[..., :n, :], l.values[..., :n, :]
    assert cache.get_seq_length() == n, "could not restore the cached prefix"

@torch.no_grad()
def prefill(spec):
    x, pos3, vm = _seq(spec)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    cache = _new_cache()
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        body(inputs_embeds=x, position_ids=pos3, visual_pos_masks=vm[None].to(EMB_DEVICE),
             deepstack_visual_embeds=spec["ds"], past_key_values=cache, use_cache=True)
    _sync()
    return {"cache": cache, "n": int(x.shape[1]), "next_pos": spec["next_pos"], "sec": time.time() - t0,
            "nvis": int(spec["nvis"]), "peak": max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9}

@torch.no_grad()
def suffix_pass(P, suf_ids, n_opt):
    E = body.embed_tokens
    suf = E(suf_ids.to(EMB_DEVICE))[None]
    n_q = suf.shape[1]
    pos3 = (P["next_pos"] + torch.arange(n_q, device=EMB_DEVICE)).view(1, 1, -1).expand(3, 1, -1)
    am = torch.ones(1, P["n"] + n_q, dtype=torch.long, device=EMB_DEVICE)
    _sync(); t0 = time.time()
    with sdpa_kernel([SDPBackend.MATH]):
        out = body(inputs_embeds=suf, attention_mask=am, position_ids=pos3, past_key_values=P["cache"], use_cache=True)
    h = out[0][0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    _crop(P["cache"], P["n"])
    return _result(logits, n_opt, {"ntok": P["nvis"], "vis_mean_layers": float(P["nvis"]), "sec": float(P["sec"] + dt),
                                   "sec_suffix": float(dt), "flops": llm_flops(P["n"] + n_q),
                                   "peak_gb": round(P["peak"], 3)})

@torch.no_grad()
def preflight_attention(n_tok):
    dev = body.layers[0].self_attn.q_proj.weight.device
    q = torch.randn(1, NHEADS, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    kv = torch.randn(1, NKV, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    mask = torch.ones(n_tok, n_tok, dtype=torch.bool, device=dev).tril()[None, None]
    torch.cuda.reset_peak_memory_stats(dev)
    base = torch.cuda.memory_allocated(dev)
    try:
        with sdpa_kernel(NO_MATH):
            o1 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, is_causal=True, enable_gqa=True)
            o2 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, attn_mask=mask, enable_gqa=True)  # routed to is_causal
        torch.cuda.synchronize(dev)          # surface any asynchronous CUDA error HERE, not in a later cell
        assert torch.equal(o1, o2), "causal-mask routing changed the result"
    except RuntimeError as e:
        raise RuntimeError(f"Memory-efficient attention is not available here ({e}). Send me this message.")
    del o1, o2
    peak = (torch.cuda.max_memory_allocated(dev) - base) / 1e9
    del q, kv, mask
    torch.cuda.empty_cache()
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU{i}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    print(f"Attention preflight OK ✅  {n_tok} tokens -> extra peak memory {peak:.2f} GB")

preflight_attention(NUM_FRAMES // 2 * (MAX_PIXELS // FACTOR ** 2) + 600)
print("Model ready ✅")

### 4b · Safety checks (run on the first video)

In [ ]:
# ── Safety check 1: our pipeline must reproduce the official model forward (processor + Qwen3-VL forward) ──
def equivalence_check(frame_sets, row):
    report = {"transformers": transformers.__version__}
    try:
        frames, meta = frame_sets["orig"], frame_sets["_meta"]["orig"]
        n_opt = min(len(row.options), len(LETTERS))
        inp = prepare_video(frames, meta, prompt_tail=suffix_text(row.question, row.options))
        dev_in = {k: (v.to(VDEV, torch.float16) if torch.is_floating_point(v) else v.to(EMB_DEVICE)) for k, v in inp.items()}
        with torch.no_grad(), sdpa_kernel(NO_MATH):
            try:
                out = model(**dev_in, logits_to_keep=1)
            except TypeError:
                out = model(**dev_in, num_logits_to_keep=1)
        lp_off = torch.log_softmax(out.logits[0, -1].float()[LETTER_IDS[:n_opt]], -1).cpu().numpy()
        del out, dev_in
        enc = encode_frames(frames, meta)
        spec = full_spec(enc)
        suf = build_suffix_ids(row.question, row.options)
        report["prompt_tokens_identical"] = bool(torch.equal(inp["input_ids"][0], torch.cat([enc["ids"], suf])))
        report["n_video_tokens"] = int(enc["n_full"])
        report["temporal_groups"] = int(enc["G"]); report["tokens_per_group"] = int(enc["P"])
        ours = llm_pass(spec, suf, n_opt)
        man = llm_pass(spec, suf, n_opt, manual=True)
        report["logp_official"] = lp_off.round(4).tolist()
        report["logp_ours"] = ours["logp"]
        report["max_abs_gap"] = float(np.abs(lp_off - np.array(ours["logp"])).max())
        report["logp_ours_manual_loop"] = man["logp"]
        report["max_abs_gap_manual_loop"] = float(np.abs(np.array(man["logp"]) - np.array(ours["logp"])).max())
        report["full_model_answer"] = ours["pred"]
        ok = report["prompt_tokens_identical"] and report["max_abs_gap"] <= 0.15 and report["max_abs_gap_manual_loop"] <= 0.05
        report["status"] = "PASS" if ok else "FAIL"
    except Exception as e:
        report["status"] = "SKIPPED"
        report["error"] = f"{type(e).__name__}: {e}"
    with open(os.path.join(OUT_DIR, "equivalence_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Equivalence check:", json.dumps(report, indent=2))
    if report["status"] != "PASS":
        raise RuntimeError("Equivalence check failed — stopping so no GPU hours are wasted. "
                           "Send me equivalence_check.json and the log.")
    gc.collect(); torch.cuda.empty_cache()


# ── Safety check 2: prefix caching must give the same answers as running each pass in full ──
def cache_check(enc, comp, rows):
    report = {"checks": []}
    picks = [("full", None)] + [k for k in comp if k in (("random", "0.1"), ("uniform", "0.1"))][:2]
    for key in picks:
        spec = full_spec(enc) if key[0] == "full" else comp[key]
        P = prefill(spec)
        for row in rows[:2]:
            n_opt = min(len(row.options), len(LETTERS))
            for opts in (row.options, reorder_options(row.options, "option_reverse")):
                suf = build_suffix_ids(row.question, opts)
                a = llm_pass(spec, suf, n_opt)
                b = suffix_pass(P, suf, n_opt)
                report["checks"].append({"pass": "/".join(map(str, key)) if key[0] != "full" else "full",
                                         "qid": row.qid, "same_answer": a["pred"] == b["pred"],
                                         "gap": float(np.abs(np.array(a["logp"]) - np.array(b["logp"])).max()),
                                         "speedup_per_extra_question": round(a["sec"] / max(b["sec_suffix"], 1e-6), 1)})
        del P
    same = np.mean([c["same_answer"] for c in report["checks"]])
    report["max_gap"] = max(c["gap"] for c in report["checks"])
    report["same_answer_share"] = float(same)
    report["status"] = "PASS" if (same == 1.0 and report["max_gap"] <= 0.15) else "FAIL"
    with open(os.path.join(OUT_DIR, "cache_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Cache check:", report["status"], f"| max log-prob gap {report['max_gap']:.4f} | "
          f"same answers {100 * same:.0f}% | an extra question on a cached prefix is "
          f"~{np.median([c['speedup_per_extra_question'] for c in report['checks']]):.0f}x faster than a full pass")
    if report["status"] != "PASS":
        raise RuntimeError("Cache check failed — stopping so no GPU hours are wasted. Send me cache_check.json.")
    gc.collect(); torch.cuda.empty_cache()

### 5 · Method code (FlashVID, HoliTom, VisionZip — same ports as the main study)

In [ ]:
# ── FlashVID, HoliTom and VisionZip: the same faithful ports used in the main study (see that notebook for the
#    line-by-line provenance). Only additions here: FlashVID's ADTS v1 token selection ("attn_div"), which its
#    Qwen3-VL script uses, and a HoliTom helper that assigns the merged tokens of each window to frames, needed for
#    LLaVA-Video's per-frame newline token. ──
# ══ Official method code shared by the replication models (same ports as the main study) ══
# Each compressor returns a dict: tokens (K, D) entering the LLM, pos (their position ids, contiguous for all three
# as in the official code), newline (whether LLaVA-OV's video newline token is appended), and inner (an optional
# inner-LLM operation applied by llm_pass at a given layer).

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# FlashVID (Fan et al., ICLR 2026 oral) — github.com/Fanziyang-v/FlashVID, flashvid/utils.py + token_selection.py
# MIT License, Copyright (c) the FlashVID authors. The functions below are copied verbatim (only the config class is
# re-declared here); the official LLaVA-OV path passes the bilinear-pooled features and the bilinear-pooled
# last-vision-layer attention (mean over heads and queries), then prunes inside the LLM at layer 20 (FastV-style,
# last-query attention of layer 19, keep 30%). Positions are contiguous before the LLM, original after pruning.
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Optional, Tuple, Union, List

@dataclass
class FlashVidConfig:
    retention_ratio: float = 0.25
    alpha: float = 0.7
    token_selection_method: str = "attn_div_v2"
    temporal_threshold: float = 0.8
    do_segment: bool = True
    segment_threshold: float = 0.9
    min_segment_num: int = 8
    complementary_segment: bool = True
    num_attn_div_tokens: Optional[int] = None
    num_sttm_tokens: Optional[int] = None
    visual_token_start_index: Optional[int] = None
    visual_token_length: Optional[int] = None
    expansion: float = 1.25
    pruning_layer: int = 20
    llm_retention_ratio: float = 0.3

def pairwise_cosine_distances(image_features):
    normed_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    similarities = torch.bmm(normed_features, normed_features.transpose(-1, -2))
    return 1.0 - similarities

def attn_div_v2_based_token_selection(features, cls_attention, num_retained_tokens):
    original_features = features
    features = features.float()
    pooled_features = features.mean(1)
    global_cls_attention = cls_attention.float() * 1e6
    bsz, num_visual_tokens, feat_dim = features.shape
    dist_matrix = pairwise_cosine_distances(features)
    calibration_term1 = global_cls_attention.unsqueeze(1)
    local_cls_attention = torch.einsum("b n d, c d -> b c n", features, pooled_features).mean(1)
    calibration_term2 = local_cls_attention.unsqueeze(1)
    dist_matrix = dist_matrix * calibration_term1 * calibration_term2
    keep_indices = torch.zeros(bsz, num_retained_tokens, dtype=torch.long, device=features.device)
    min_dist = torch.topk(dist_matrix, k=2, dim=1, largest=False).values[:, 1, :]
    keep_indices[:, 0] = torch.argmax(min_dist, dim=-1)
    for i in range(1, num_retained_tokens):
        dist_sub_matrix = torch.gather(dist_matrix, dim=1,
                                       index=keep_indices[:, :i].unsqueeze(-1).expand(-1, -1, num_visual_tokens))
        min_dist = torch.min(dist_sub_matrix, dim=1).values
        keep_indices[:, i] = torch.argmax(min_dist, dim=-1)
    keep_indices = keep_indices.sort().values
    selected_features = torch.gather(original_features, dim=1,
                                     index=keep_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, keep_indices

def attn_based_token_selection(features, cls_attention, num_retained_tokens):
    bsz, num_visual_tokens, feat_dim = features.shape
    topk_indices = torch.topk(cls_attention, k=num_retained_tokens, dim=-1).indices.sort().values
    selected_features = torch.gather(features, dim=1, index=topk_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, topk_indices

def attn_div_based_token_selection(features, cls_attention, num_retained_tokens):
    # FlashVID ADTS v1 (token_selection.py), used by its Qwen3-VL script; copied verbatim
    original_features = features
    features = features.float()
    cls_attention = cls_attention.float() * 1e6
    bsz, num_visual_tokens, feat_dim = features.shape
    dist_matrix = pairwise_cosine_distances(features)
    dist_matrix = dist_matrix * cls_attention.unsqueeze(1)
    keep_indices = torch.zeros(bsz, num_retained_tokens, dtype=torch.long, device=features.device)
    min_dist = torch.topk(dist_matrix, k=2, dim=1, largest=False).values[:, 1, :]
    keep_indices[:, 0] = torch.argmax(min_dist, dim=-1)
    for i in range(1, num_retained_tokens):
        dist_sub_matrix = torch.gather(dist_matrix, dim=1,
                                       index=keep_indices[:, :i].unsqueeze(-1).expand(-1, -1, num_visual_tokens))
        min_dist = torch.min(dist_sub_matrix, dim=1).values
        min_dist.scatter_(1, keep_indices[:, :i], -1)
        keep_indices[:, i] = torch.argmax(min_dist, dim=-1)
    keep_indices = keep_indices.sort().values
    selected_features = torch.gather(original_features, dim=1,
                                     index=keep_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, keep_indices

ALL_TOKEN_SELECTION_METHOD = {"attn_div_v2": attn_div_v2_based_token_selection, "attn_div": attn_div_based_token_selection,
                              "attn": attn_based_token_selection}

def flashvid_compression(video_features, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    if flashvid_config.do_segment:
        segment_lengths = fv_segment(video_features=video_features.mean(1),
                                     segment_threshold=flashvid_config.segment_threshold,
                                     min_segment_num=flashvid_config.min_segment_num,
                                     complementary_segment=flashvid_config.complementary_segment)
    else:
        segment_lengths = torch.tensor([num_frames], dtype=torch.long, device=video_features.device)
    num_segments = segment_lengths.shape[0]
    global_indices = torch.arange(num_frames * num_visual_tokens, dtype=torch.long, device=video_features.device)
    token_budget = math.ceil(num_visual_tokens * flashvid_config.retention_ratio * flashvid_config.expansion)
    num_attn_div_tokens = math.ceil(token_budget * flashvid_config.alpha)
    num_sttm_tokens = token_budget - num_attn_div_tokens
    flashvid_config.num_attn_div_tokens = num_attn_div_tokens
    flashvid_config.num_sttm_tokens = num_sttm_tokens
    all_segment_features, all_segment_indices = [], []
    offset = 0
    for seg_idx in range(num_segments):
        seg_len = segment_lengths[seg_idx]
        segment_features = video_features[offset: offset + seg_len]
        segment_cls_attention = cls_attention[offset: offset + seg_len]
        segment_global_indices = global_indices.view(num_frames, num_visual_tokens)[offset: offset + seg_len]
        segment_features, segment_global_indices = fv_segment_compression(
            segment_features=segment_features, segment_global_indices=segment_global_indices,
            cls_attention=segment_cls_attention, flashvid_config=flashvid_config)
        all_segment_features.append(segment_features)
        all_segment_indices.append(segment_global_indices)
        offset += seg_len
    final_tokens = torch.cat(all_segment_features, dim=0)
    final_global_indices = torch.cat(all_segment_indices, dim=0)
    sorted_indices = final_global_indices.argsort()
    sorted_tokens = final_tokens[sorted_indices]
    flashvid_config.visual_token_length = sorted_tokens.shape[0]
    return sorted_tokens, final_global_indices[sorted_indices]

def fv_segment_compression(segment_features, segment_global_indices, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = segment_features.shape
    if flashvid_config.alpha > 0:
        additional_kwargs = {"cls_attention": cls_attention} if "attn" in flashvid_config.token_selection_method else {}
        selected_features, selected_indices = ALL_TOKEN_SELECTION_METHOD[flashvid_config.token_selection_method](
            features=segment_features, num_retained_tokens=flashvid_config.num_attn_div_tokens, **additional_kwargs)
        selected_global_indices = segment_global_indices.gather(1, index=selected_indices).view(-1)
    else:
        selected_features = torch.tensor([]).to(segment_features)
        selected_indices = torch.tensor([]).to(segment_global_indices)
        selected_global_indices = torch.tensor([]).to(segment_global_indices)
    mask = torch.ones(num_frames, num_visual_tokens, dtype=torch.bool, device=segment_features.device)
    mask.scatter_(1, selected_indices, False)
    num_other_tokens = flashvid_config.num_sttm_tokens * num_frames
    if num_other_tokens > 0 and flashvid_config.temporal_threshold < 1.0:
        if num_frames > 1:
            temp_merged_token_list, temp_merged_indices_list = fv_spatiotemporal_compression(
                video_features=segment_features, temporal_threshold=flashvid_config.temporal_threshold,
                token_mask=mask, flashvid_config=flashvid_config)
            temp_merged_global_indices_list = [segment_global_indices.view(num_frames, -1)[i][temp_merged_indices]
                                               for i, temp_merged_indices in enumerate(temp_merged_indices_list)]
        else:
            temp_merged_token_list = [segment_features[0]]
            temp_merged_global_indices_list = [segment_global_indices[0]]
    else:
        temp_merged_token_list = []
        temp_merged_global_indices_list = []
    all_tokens = [selected_features.view(-1, feat_dim)]
    all_global_indices = [selected_global_indices]
    if num_other_tokens > 0:
        num_current_retained_tokens = sum(len(tokens) for tokens in temp_merged_token_list)
        adapative_contextual_ratio = num_other_tokens / num_current_retained_tokens
        if adapative_contextual_ratio < 1.0:
            num_frames_in_segment = len(temp_merged_token_list)
            max_num_tokens = max(len(tokens) for tokens in temp_merged_token_list)
            batched_tokens = torch.zeros((num_frames_in_segment, max_num_tokens, feat_dim),
                                         dtype=segment_features.dtype, device=segment_features.device)
            valid_token_mask = torch.zeros((num_frames_in_segment, max_num_tokens), dtype=torch.bool,
                                           device=segment_features.device)
            num_clusters_list, k_list = [], []
            for i, temp_merged_tokens in enumerate(temp_merged_token_list):
                num_tokens = len(temp_merged_tokens)
                batched_tokens[i, :num_tokens] = temp_merged_tokens
                valid_token_mask[i, :num_tokens] = True
                num_clusters = math.ceil(num_tokens * adapative_contextual_ratio)
                num_clusters_list.append(num_clusters)
                k_list.append(min(num_clusters, 7))
            cluster_indices_list, cluster_center_indices_list = fv_dpc_knn(
                features=batched_tokens, num_clusters=num_clusters_list, k=k_list, valid_token_mask=valid_token_mask)
            for i, (temp_merged_tokens, temp_merged_global_indices) in enumerate(
                    zip(temp_merged_token_list, temp_merged_global_indices_list)):
                num_clusters = num_clusters_list[i]
                if num_clusters > 0:
                    cluster_indices = cluster_indices_list[i][:len(temp_merged_tokens)]
                    cluster_center_indices = cluster_center_indices_list[i]
                    aggregated_tokens = torch.zeros((num_clusters, feat_dim), dtype=segment_features.dtype,
                                                    device=segment_features.device)
                    aggregated_tokens.scatter_add_(0, cluster_indices.unsqueeze(-1).expand(-1, feat_dim),
                                                   temp_merged_tokens)
                    cluster_counts = torch.bincount(cluster_indices, minlength=num_clusters).unsqueeze(-1).to(
                        segment_features.dtype)
                    aggregated_tokens = aggregated_tokens / cluster_counts
                    global_token_indices = temp_merged_global_indices[cluster_center_indices]
                else:
                    aggregated_tokens = temp_merged_tokens
                    global_token_indices = temp_merged_global_indices
                all_tokens.append(aggregated_tokens)
                all_global_indices.append(global_token_indices)
        else:
            for temp_merged_tokens, temp_merged_global_indices in zip(temp_merged_token_list,
                                                                      temp_merged_global_indices_list):
                all_tokens.append(temp_merged_tokens)
                all_global_indices.append(temp_merged_global_indices)
    segment_final_tokens = torch.cat(all_tokens, dim=0)
    segment_final_global_indices = torch.cat(all_global_indices, dim=0)
    return segment_final_tokens, segment_final_global_indices

def fv_segment(video_features, segment_threshold, min_segment_num, complementary_segment=True):
    num_frames, feat_dim = video_features.shape
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    transition_similarities = torch.sum(normed_video_features[:-1] * normed_video_features[1:], dim=-1)
    cut_indices = torch.where(transition_similarities < segment_threshold)[0]
    return fv_additional_segment(cut_indices=cut_indices, num_frames=num_frames, min_segment_num=min_segment_num,
                                 transition_similarities=transition_similarities,
                                 segment_threshold=segment_threshold, complementary_segment=complementary_segment)

def fv_additional_segment(cut_indices, num_frames, min_segment_num, transition_similarities, segment_threshold,
                          complementary_segment=True):
    num_segments = cut_indices.numel() + 1
    if num_segments < min_segment_num and complementary_segment:
        num_remaining_cut_indices = min_segment_num - num_segments
        transition_similarities[transition_similarities < segment_threshold] = 1.0
        complementary_cut_indices = torch.topk(transition_similarities,
                                               k=min(num_remaining_cut_indices, transition_similarities.shape[0]),
                                               largest=False).indices
        cut_indices = torch.cat([cut_indices, complementary_cut_indices]).sort().values
    padded_cut_indices = F.pad(cut_indices, (1, 1), value=0)
    padded_cut_indices[0] = -1
    padded_cut_indices[-1] = num_frames - 1
    return torch.diff(padded_cut_indices, n=1, dim=0)

@torch.no_grad()
def fv_dpc_knn(features, num_clusters, k=7, valid_token_mask=None):
    invalid_token_mask = ~valid_token_mask if valid_token_mask is not None else None
    bsz, seq_len, feat_dim = features.shape
    dists = torch.cdist(features.float(), features.float()) / math.sqrt(feat_dim)
    if valid_token_mask is not None:
        dists = torch.masked_fill(dists, invalid_token_mask.unsqueeze(1).expand(-1, seq_len, -1), dists.max() + 1)
    max_k = max(k) if isinstance(k, list) else k
    nearest_dist = torch.topk(dists, k=max_k, dim=-1, largest=False).values
    if isinstance(k, list):
        density = torch.empty((bsz, seq_len), device=features.device, dtype=features.dtype)
        for i in range(bsz):
            density[i] = torch.mean(-(nearest_dist[i, :, :k[i]] ** 2), dim=-1).exp()
    else:
        density = torch.mean(-(nearest_dist ** 2), dim=-1).exp()
    if valid_token_mask is not None:
        density = torch.masked_fill(density, invalid_token_mask, 0.0)
    mask = density[:, None, :] > density[:, :, None]
    max_dist = dists.view(bsz, -1).max(dim=-1)[0].view(-1, 1, 1)
    modified_dists = torch.where(mask, dists, max_dist)
    dist, _ = torch.min(modified_dists, dim=-1)
    score = dist * density
    if isinstance(num_clusters, int):
        cluster_center_indices = torch.topk(score, k=num_clusters, dim=-1).indices
        dists = torch.gather(dists, dim=-1, index=cluster_center_indices.unsqueeze(1).expand(-1, seq_len, -1))
        cluster_indices = torch.argmin(dists, dim=-1)
        cluster_indices.scatter_(dim=-1, index=cluster_center_indices,
                                 src=torch.arange(num_clusters).to(cluster_indices).unsqueeze(0).expand(bsz, -1))
        return cluster_indices, cluster_center_indices
    cluster_indices_list, cluster_center_indices_list = [], []
    for i in range(bsz):
        k_i = num_clusters[i]
        if k_i == 0:
            cluster_center_indices_list.append(torch.tensor([], dtype=torch.long, device=features.device))
            cluster_indices_list.append(torch.zeros(seq_len, dtype=torch.long, device=features.device))
            continue
        cc_idx = torch.topk(score[i], k=k_i, dim=-1).indices
        cluster_center_indices_list.append(cc_idx)
        dists_i = torch.gather(dists[i], dim=-1, index=cc_idx.unsqueeze(0).expand(seq_len, -1))
        c_idx = torch.argmin(dists_i, dim=-1)
        c_idx.scatter_(dim=-1, index=cc_idx, src=torch.arange(k_i).to(c_idx))
        cluster_indices_list.append(c_idx)
    return cluster_indices_list, cluster_center_indices_list

def fv_spatiotemporal_compression(video_features, temporal_threshold, token_mask, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    lower_bound = (flashvid_config.num_attn_div_tokens + flashvid_config.num_sttm_tokens) * num_frames
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    cosine_similarities = torch.bmm(normed_video_features[1:], normed_video_features[:-1].transpose(1, 2))
    cosine_similarities[~token_mask[1:].unsqueeze(-1).expand(-1, -1, num_visual_tokens)] = -1.0
    cosine_similarities[~token_mask[:-1].unsqueeze(1).expand(-1, num_visual_tokens, -1)] = -1.0
    max_sims, max_sim_indices = torch.max(cosine_similarities, dim=-1)
    padded_max_sims = F.pad(max_sims, (0, 0, 1, 0), value=-1)
    padded_max_sim_indices = F.pad(max_sim_indices, (0, 0, 1, 0), value=-1)
    token_counts = torch.ones(num_frames, num_visual_tokens).to(video_features)
    mask = padded_max_sims > temporal_threshold
    retaining_token_mask = ~mask
    if retaining_token_mask.int().sum() < lower_bound:
        soft_threshold = padded_max_sims.view(-1).topk(k=(num_frames * num_visual_tokens) - lower_bound).values[-1]
        soft_threshold = max(soft_threshold, -1.0 + 1e-6)
        mask = padded_max_sims > soft_threshold
        retaining_token_mask = ~mask
    for frame_idx in range(num_frames - 1, -1, -1):
        frame_features = video_features[frame_idx]
        frame_token_counts = token_counts[frame_idx]
        frame_max_sim_indices = padded_max_sim_indices[frame_idx]
        tokens_to_merge = frame_features[~mask[frame_idx]]
        to_merge_token_counts = frame_token_counts[~mask[frame_idx]]
        if tokens_to_merge.numel() > 0:
            aggregated_tokens = tokens_to_merge / to_merge_token_counts.unsqueeze(-1).to(tokens_to_merge.dtype)
            video_features[frame_idx][~mask[frame_idx]] = aggregated_tokens
            token_counts[frame_idx][~mask[frame_idx]] = 1
        other_tokens = frame_features[mask[frame_idx]]
        if other_tokens.numel() > 0:
            anchor_token_indices = frame_max_sim_indices[mask[frame_idx]]
            aggregated_tokens = torch.zeros((num_visual_tokens, feat_dim), dtype=video_features.dtype,
                                            device=video_features.device)
            aggregated_tokens.scatter_add_(0, anchor_token_indices.unsqueeze(-1).expand(-1, feat_dim), other_tokens)
            aggregated_token_counts = torch.bincount(anchor_token_indices, minlength=num_visual_tokens).to(
                video_features.dtype)
            video_features[frame_idx - 1] += aggregated_tokens
            token_counts[frame_idx - 1] += aggregated_token_counts
            token_counts[frame_idx][mask[frame_idx]] = 0
    final_tokens, retained_token_indices = [], []
    for i in range(num_frames):
        frame_mask = retaining_token_mask[i] & token_mask[i]
        final_tokens.append(video_features[i][frame_mask])
        retained_token_indices.append(torch.where(frame_mask)[0])
    return final_tokens, retained_token_indices


# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# HoliTom (Shao et al., NeurIPS 2025) — github.com/cokeshao/HoliTom, holitom/llava_arch.py + modeling_qwen2.py
# Apache-2.0. Methods copied as plain functions (self removed); settings from scripts/eval_ov-7b_holitom.sh:
# T=0.65 at 10% (0.80 otherwise), BETA=0.6 (beta weighting on, as the script leaves NO_BETA unset), D=0, K=7,
# inner-LLM merging at layer 18 with r=0.5 (HOLITOM_k=18, HOLITOM_r=0.5). Only edit: the 1e-6 density tie-break noise
# uses a seeded generator. LLaVA-OV uses mm_newline_position "one_token", so tokens are [static, dynamic] per window,
# one newline at the end, contiguous positions; the inner merge treats the newline as part of the image span (as there).
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
def ht_cluster_dpc_knn(x, cluster_num, k=7, gen=None):
    with torch.no_grad():
        batch_size, seq_len, embed_dim = x.shape
        dist_matrix = torch.cdist(x.float(), x.float()) / (embed_dim ** 0.5)
        dist_nearest, index_nearest = torch.topk(dist_matrix, k, dim=-1, largest=False)
        density = (-(dist_nearest ** 2).mean(dim=-1)).exp()
        density = density + torch.rand(density.shape, device=density.device, dtype=density.dtype, generator=gen) * 1e-6
        mask = (density[:, None, :] > density[:, :, None]).type(x.dtype)
        dist_max = dist_matrix.flatten(1).max(dim=-1).values[:, None, None]
        dist, index_parent = (dist_matrix * mask + dist_max * (1 - mask)).min(dim=-1)
        score = dist * density
        _, index_center = score.topk(cluster_num, dim=-1)
        return index_center, dist_matrix

def ht_select_static_windows(feature_sim, batch_size, tau, max_window_size):
    def get_pruned_static_count_vectorized(feature_sim, batch_size, tau):
        similarity_matrix = torch.ones((batch_size, batch_size, feature_sim.shape[1]), device=feature_sim.device)
        for start in range(batch_size - 1):
            cum_similarity = torch.cumprod(feature_sim[start:] > tau, dim=0)
            similarity_matrix[start, start + 1:start + 1 + len(cum_similarity)] = cum_similarity
        window_lengths = torch.arange(batch_size, device=feature_sim.device).unsqueeze(0) - \
            torch.arange(batch_size, device=feature_sim.device).unsqueeze(1)
        window_lengths = window_lengths.clamp(min=0)
        return (similarity_matrix.sum(dim=-1) * window_lengths).float()
    pruned_static_count = get_pruned_static_count_vectorized(feature_sim, batch_size, tau)
    dp = torch.zeros(batch_size, device=pruned_static_count.device)
    prev = torch.zeros(batch_size, dtype=torch.long, device=pruned_static_count.device)
    for i in range(batch_size):
        max_val = dp[i - 1] if i > 0 else 0
        best_j = i
        for window_size in range(2, min(i + 1, max_window_size) + 1):
            j = i - window_size
            current_val = (dp[j] if j >= 0 else 0) + pruned_static_count[j + 1, i]
            if current_val > max_val:
                max_val = current_val
                best_j = j + 1
        dp[i] = max_val
        prev[i] = best_j
    selected_frames = []
    i = batch_size - 1
    while i >= 0:
        selected_frames.append((prev[i].item(), i))
        i = prev[i].item() - 1
    selected_frames = selected_frames[::-1]
    return selected_frames, dp[-1].item()

def ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta):
    batch_size, seq_len, embed_dim = feat.shape
    all_indices = torch.arange(seq_len, device=feat.device).unsqueeze(0).expand(batch_size, -1)
    non_target_indices = torch.zeros((batch_size, seq_len - cluster_num), dtype=torch.long, device=feat.device)
    for b in range(batch_size):
        non_target_mask = ~torch.isin(all_indices[b], target_indices[b])
        non_target_indices[b] = all_indices[b][non_target_mask]
    non_target_feat = torch.gather(feat, dim=1, index=non_target_indices.unsqueeze(-1).expand(-1, -1, feat.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=1,
                               index=non_target_indices.unsqueeze(-1).expand(-1, -1, dist_matrix.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=2, index=target_indices.unsqueeze(1).expand(-1, dist_matrix.size(1), -1))
    idx_cluster = torch.argmin(dist_matrix, dim=-1)
    cluster_tokens = []
    for b in range(batch_size):
        batch_tokens = []
        for i in range(cluster_num):
            mask = (idx_cluster[b] == i)
            if mask.any():
                cluster_means = non_target_feat[b][mask].mean(dim=0)          # NO_BETA unset -> beta weighting
                batch_tokens.append(Beta * feat[b][target_indices[b][i]] + (1 - Beta) * cluster_means)
            else:
                batch_tokens.append(feat[b][target_indices[b][i]])
        cluster_tokens.append(torch.stack(batch_tokens))
    return torch.stack(cluster_tokens)

def ht_merge_tokens_by_attention_density(feat, attn, pos, retain_ratio, D, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    dominant_num = round(math.ceil(seq_len * retain_ratio) * (1 - D))
    contextual_num = math.ceil(seq_len * retain_ratio) - dominant_num
    if dominant_num > 0:
        all_indices = attn.topk(dominant_num, dim=1).indices
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device).scatter_(1, all_indices, False)
        dominant_tokens = feat.masked_select(~mask.unsqueeze(-1)).view(batch_size, dominant_num, embed_dim)
        dominant_pos = pos.masked_select(~mask).view(batch_size, dominant_num)
    else:
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device)
        dominant_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        dominant_pos = torch.empty((batch_size, 0), device=feat.device)
    if contextual_num > 0:
        feat_filtered = feat.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num, embed_dim)
        contextual_pos = pos.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num)
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat_filtered, contextual_num, k=min(K, contextual_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        contextual_pos = torch.stack([contextual_pos[b][target_indices[b]] for b in range(batch_size)])
        contextual_tokens = ht_merge_tokens_by_clustering(feat_filtered, target_indices, dist_matrix, contextual_num, Beta)
    else:
        contextual_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        contextual_pos = torch.empty((batch_size, 0), device=feat.device)
    image_feat, image_pos = [], []
    for b in range(batch_size):
        image_feat.append(torch.cat([dominant_tokens[b], contextual_tokens[b]], dim=0))
        image_pos.append(torch.cat([dominant_pos[b], contextual_pos[b]], dim=0))
    return torch.stack(image_feat), torch.stack(image_pos)

def ht_merge_tokens_by_density(feat, pos, retain_ratio, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    cluster_num = round(seq_len * retain_ratio)
    if cluster_num > 0:
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat, cluster_num, k=min(K, cluster_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        image_pos = torch.stack([pos[b][target_indices[b]] for b in range(batch_size)])
        image_feat = ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta)
    else:
        image_feat = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        image_pos = torch.empty((batch_size, 0), device=feat.device)
    return image_feat, image_pos

def ht_holitom(static_feat, dynamic_feat, dynamic_attn, static_pos, dynamic_pos, window_size, retain_ratio, D, Beta,
               K, images_dtype, gen=None):
    # mm_newline_position == "one_token" branch (LLaVA-OneVision)
    if window_size == 1:
        dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                         retain_ratio, D, Beta, K, gen)
        return dynamic_feat.flatten(0, 1).to(images_dtype)
    dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                     retain_ratio, D, Beta, K, gen)
    static_feat, static_pos = ht_merge_tokens_by_density(static_feat.unsqueeze(0), static_pos, retain_ratio, Beta, K, gen)
    return torch.cat([static_feat.flatten(0, 1), dynamic_feat.flatten(0, 1)]).to(images_dtype)

def ht_get_static_dynamic_features(image_feat, attn_weights, selected_frames, feature_sim, tau):
    batch_size, seq_len, embed_dim = image_feat.shape
    static_feat_list, dynamic_feat_list, dynamic_attn_list, static_pos_list, dynamic_pos_list = [], [], [], [], []
    for start, end in selected_frames:
        all_indices = torch.arange(seq_len, device=image_feat.device).unsqueeze(0)
        if start == end:
            static_feat_list.append(torch.empty((0, embed_dim), device=image_feat.device))
            dynamic_feat_list.append(image_feat[start:end + 1])
            dynamic_attn_list.append(attn_weights[start:end + 1])
            static_pos_list.append(torch.empty((0, seq_len), device=image_feat.device))
            dynamic_pos_list.append(all_indices)
        else:
            windows_size = end - start + 1
            mask = torch.all(feature_sim[start:end, :] > tau, dim=0)
            static_feat_list.append(image_feat[start:end + 1, mask].mean(dim=0))
            dynamic_feat_list.append(image_feat[start:end + 1, ~mask])
            dynamic_attn_list.append(attn_weights[start:end + 1, ~mask])
            static_pos_list.append(all_indices[:, mask].expand(1, -1))
            dynamic_pos_list.append(all_indices[:, ~mask].expand(windows_size, -1))
    return static_feat_list, dynamic_feat_list, None, dynamic_attn_list, static_pos_list, dynamic_pos_list


# ── VisionZip (Yang et al., CVPR 2025) — LLaVA-OV video port from the HoliTom repo (visionzip/llava_arch.py) ──
# Per frame, after LLaVA-OV's 2x2 pooling: dominant tokens by received vision attention, contextual tokens by merging
# the rest onto evenly spaced targets using key similarity. Default dominant ratio 54/64 (as in that code).
# Output order per frame = [dominant (spatial order), contextual]; frames concatenated; contiguous positions.
@torch.no_grad()
def visionzip_compress(X, vz_attn, vz_metric, r, dominant_ratio=54 / 64):
    T, S, D = X.shape
    spatial = int(S * r)
    dom = int(spatial * dominant_ratio)
    ctx = spatial - dom
    hs = X.float()
    att = vz_attn.to(hs.device).float()
    met = vz_metric.to(hs.device).float()
    topk = att.topk(dom, dim=1).indices
    mask = torch.ones(T, S, dtype=torch.bool, device=hs.device).scatter_(1, topk, False)   # True = not dominant
    dominant = hs.masked_select(~mask.unsqueeze(-1)).view(T, dom, D)
    met_f = met[mask].view(T, S - dom, met.shape[2])
    hs_f = hs.masked_select(mask.unsqueeze(-1)).view(T, S - dom, D)
    met_n = met_f / met_f.norm(dim=-1, keepdim=True)
    step = max(1, (S - dom) // ctx)
    tgt = torch.arange(0, S - dom, step, device=hs.device)[:ctx]
    tgt_tok = met_n[:, tgt, :]
    rest = ~torch.isin(torch.arange(met_n.shape[1], device=hs.device), tgt)
    to_merge = met_n[:, rest, :]
    sim = torch.bmm(to_merge, tgt_tok.transpose(1, 2))
    onehot = torch.zeros(T, to_merge.shape[1], ctx, device=hs.device)
    onehot.scatter_(2, sim.argmax(dim=2).unsqueeze(-1), 1)
    counts = onehot.sum(dim=1).clamp(min=1).unsqueeze(-1)
    agg = torch.bmm(onehot.transpose(1, 2), hs_f[:, rest, :]) / counts
    contextual = hs_f[:, tgt, :] + agg
    return torch.cat([dominant, contextual], dim=1).reshape(-1, D).to(X.dtype)

### 5b · Compressors for this model (+ self-test)

In [ ]:
# ══ Compressors for Qwen3-VL (16 temporal groups of merged tokens, 3-D positions, DeepStack) ══
# Every method returns a spec (the assembled prefix). Kept tokens keep their own 3-D positions and DeepStack features.
# FlashVID follows its official Qwen3-VL code: compression over the temporal groups (each group = one "frame" for
# FlashVID), merged tokens placed at their anchor token's position, vision start/end tokens dropped, then FastV-style
# pruning to 10% at LLM layer 28. Uniform pooling averages tokens, DeepStack features and positions over each window.

def _best_grid_hw(n, H, W):
    best = None
    for h in range(1, H + 1):
        for w in range(1, W + 1):
            key = (abs(h * w - n), abs(h / max(w, 1) - H / W), -h * w)
            if best is None or key < best[0]:
                best = (key, h, w)
    return best[1], best[2]

def compress_all(enc, seed, only=None, skip=()):
    G, P, Hm, Wm = enc["G"], enc["P"], enc["Hm"], enc["Wm"]
    X = enc["X"].view(G, P, -1)
    DS = [d.view(G, P, -1) for d in enc["DS"]]
    gp = _group_pos(enc)                                      # (3, G, P)
    dev = X.device
    jobs = []
    for r in METHOD_GRID.get("flashvid", []):
        def fv(r=r):
            cfg = FlashVidConfig(retention_ratio=r, **FLASHVID)
            toks, idx = flashvid_compression(X.clone(), enc["CA"].to(dev, X.dtype), cfg)
            g_of = idx // P
            flat_ds = [d.reshape(G * P, -1) for d in DS]
            flat_pos = gp.reshape(3, G * P)
            groups = []
            for g in range(G):
                sel = g_of == g
                gi = idx[sel]
                groups.append((toks[sel], [d[gi] for d in flat_ds], flat_pos[:, gi.cpu()]))
            s = assemble(enc, groups, drop_start_end=True)
            s.update(inner={"kind": "fastv", "layer": cfg.pruning_layer, "keep_ratio": cfg.llm_retention_ratio},
                     nvis=int(toks.shape[0]))
            return s
        jobs.append(("flashvid", str(r), fv))
    for r in METHOD_GRID.get("uniform", []):
        def un(r=r):
            h, w = _best_grid_hw(int(P * r), Hm, Wm)
            pool = lambda t: F.adaptive_avg_pool2d(t.float().reshape(G, Hm, Wm, -1).permute(0, 3, 1, 2), (h, w)) \
                .permute(0, 2, 3, 1).reshape(G, h * w, -1)
            fx = pool(X).to(X.dtype)
            fds = [pool(d).to(d.dtype) for d in DS]
            fpos = F.adaptive_avg_pool2d(gp.permute(1, 0, 2).reshape(G, 3, Hm, Wm).float(), (h, w)).round().long()
            fpos = fpos.reshape(G, 3, h * w).permute(1, 0, 2)
            s = assemble(enc, [(fx[g], [d[g] for d in fds], fpos[:, g]) for g in range(G)])
            s.update(inner=None, nvis=G * h * w)
            return s
        jobs.append(("uniform", str(r), un))
    for r in METHOD_GRID.get("random", []):
        def rd(r=r):
            k = int(P * r)
            gen = torch.Generator(device="cpu").manual_seed(seed)
            idx = torch.stack([torch.randperm(P, generator=gen)[:k].sort().values for _ in range(G)])
            groups = [(X[g][idx[g].to(dev)], [d[g][idx[g].to(dev)] for d in DS], gp[:, g][:, idx[g]]) for g in range(G)]
            s = assemble(enc, groups)
            s.update(inner=None, nvis=G * k)
            return s
        jobs.append(("random", str(r), rd))
    out, failures = {}, []
    for m, s, fn in jobs:
        if (only is not None and m not in only) or m in skip:
            continue
        try:
            out[(m, s)] = fn()
        except Exception as e:
            failures.append((m, s, f"{type(e).__name__}: {e}"))
    return out, failures

# ── self-test on a synthetic "video" (shapes, finiteness, budgets) ──
def _fake_enc(G=16, Hm=10, Wm=18, D=64, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    P = Hm * Wm
    base = torch.randn(1, P, D, generator=g)
    X = (base + torch.cumsum(0.25 * torch.randn(G, 1, D, generator=g), 0) + 0.6 * torch.randn(G, P, D, generator=g))
    pre = 10
    ids, pos, cur = [7] * pre, [list(range(pre))] * 3, pre
    ids, pos = list(ids), [list(p) for p in pos]
    for gi in range(G):
        ids += [5, VSTART_ID] + [VIDEO_TOKEN_ID] * P + [VEND_ID]
        for r in range(3):
            pos[r] += [cur, cur + 1]
        cur += 2
        hh, ww = torch.meshgrid(torch.arange(Hm), torch.arange(Wm), indexing="ij")
        pos[0] += [cur] * P; pos[1] += (cur + hh.flatten()).tolist(); pos[2] += (cur + ww.flatten()).tolist()
        cur += max(Hm, Wm)
        for r in range(3):
            pos[r] += [cur]
        cur += 1
    ids = torch.tensor(ids); pos = torch.tensor(pos)
    return {"ids": ids, "pos": pos, "vmask": ids == VIDEO_TOKEN_ID, "X": X.reshape(G * P, D).to(EMB_DEVICE, torch.float16),
            "DS": [torch.randn(G * P, D).to(EMB_DEVICE, torch.float16) for _ in range(N_DEEPSTACK)],
            "CA": torch.rand(G, P).to(EMB_DEVICE), "G": G, "Hm": Hm, "Wm": Wm, "P": P,
            "text_emb": torch.randn(len(ids), D).to(EMB_DEVICE, torch.float16), "next_pos": int(pos.max()) + 1,
            "n_full": G * P}

_enc = _fake_enc()
_comp, _fails = compress_all(_enc, seed=0)
assert not _fails, _fails
assert len(_comp) == sum(len(v) for v in METHOD_GRID.values()), "missing compressed passes"
for (m, s), c in _comp.items():
    assert torch.isfinite(c["embeds"].float()).all(), (m, s)
    assert c["embeds"].shape[0] == c["pos"].shape[1] == c["vmask"].shape[0], (m, s)
    assert int(c["vmask"].sum()) == c["nvis"] == c["ds"][0].shape[0], (m, s)
    print(f"  {m:9s} {s:5s} enters LLM with {c['nvis']:5d} visual tokens = {c['nvis'] / _enc['n_full']:.3f}"
          + (f" | inner {c['inner']['kind']} @ layer {c['inner']['layer']}" if c["inner"] else ""))
del _enc, _comp
print(f"Compressor self-test OK ✅  ({sum(len(v) for v in METHOD_GRID.values())} compressed passes)")

### 6 · Main run

In [ ]:
# ── Main run: per video encode 3 frame sets, compress once, cache each prefix once, run every question on it ──
import zlib
done = set()
if os.path.exists(RES_PATH):
    with open(RES_PATH) as f:
        for line in f:
            try:
                done.add(json.loads(line)["qid"])
            except Exception:
                pass
print(f"{len(done)} questions already in results (resume)")

table = build_manifest()
T_START = time.time()
time_left = lambda: MAX_RUNTIME_HOURS * 3600 - (time.time() - T_START)

def video_groups():
    """Yields (video_key, local path, DataFrame of its questions still to run)."""
    if BENCHMARK == "videomme":
        finished = []                    # a chunk folder is deleted only once none of its videos can still be in use
        for ch in VIDEOMME_CONFIRM_CHUNKS:
            if time_left() < 1800:
                print("⏱️ Time budget reached — skipping remaining chunks"); return
            vids = videomme_fetch_chunk(ch)
            sub = table[(table.video_key.isin(vids.keys()) | table.video_alt.isin(vids.keys())) & ~table.qid.isin(done)]
            if QUICK_MAX_Q:
                sub = sub[sub.video_key.isin(sub.video_key.drop_duplicates().head(max(1, QUICK_MAX_Q // 3)))]
            if sub.empty:
                shutil.rmtree(os.path.join(WORK_DIR, f"vmme_{ch}"), ignore_errors=True)
                continue
            while len(finished) > 1:     # the main loop may still be reading the last video of the previous chunk
                shutil.rmtree(os.path.join(WORK_DIR, f"vmme_{finished.pop(0)}"), ignore_errors=True)
            print(f"Chunk {ch}: {len(sub)} questions, {sub.video_key.nunique()} videos")
            for vk, g in sub.groupby("video_key", sort=False):
                yield vk, vids.get(vk) or vids.get(g.video_alt.iloc[0]), g
            finished.append(ch)
            if QUICK_MAX_Q:
                return                                   # quick test: the first chunk with sample videos is enough
    else:
        sub = table[~table.qid.isin(done)]
        if QUICK_MAX_Q:
            per = max(1, QUICK_MAX_Q // max(1, sub.group.nunique()))
            sub = sub.groupby("group", group_keys=False).head(per).head(QUICK_MAX_Q)
        keys = sorted(sub.video_key.unique())
        print(f"{len(sub)} questions on {len(keys)} videos to run; downloading videos…")
        fetch = {"egoschema": egoschema_fetch, "longvideobench": longvideobench_fetch}[BENCHMARK]
        paths = fetch(keys)
        missing = [k for k in keys if k not in paths]
        for k in missing:
            with open(ERR_PATH, "a") as f:
                f.write(f"{k}\tmissing video\t{len(sub[sub.video_key == k])} questions dropped\n")
        if missing:
            print(f"⚠️ {len(missing)} videos could not be found/downloaded (listed in errors.log)")
        for vk, g in sub[sub.video_key.isin(paths.keys())].groupby("video_key", sort=False):
            yield vk, paths[vk], g

def _finish(res, row):
    res["correct"] = bool(res["pred"] == row.answer)
    return res

def run_video(fsets, rows, seed, first):
    enc = encode_frames(fsets["orig"], fsets["_meta"]["orig"])
    n_full = enc["n_full"]
    comp, fails = compress_all(enc, seed)
    if first:
        cache_check(enc, comp, rows)
    recs = {}
    for row in rows:
        recs[row.qid] = {"qid": row.qid, "video": row.video_key, "group": row.group, "task": row.task,
                         "gt": row.answer, "n_opt": min(len(row.options), len(LETTERS)), "n_full": n_full,
                         "noise": {}, "methods": {}}
    sufs = {row.qid: build_suffix_ids(row.question, row.options) for row in rows}
    # full model + option-order noise share one prefix
    P = prefill(full_spec(enc))
    for row in rows:
        n = recs[row.qid]["n_opt"]
        recs[row.qid]["full"] = _finish(suffix_pass(P, sufs[row.qid], n), row)
        for kind in NOISE_OPTION_ORDERS:
            res = suffix_pass(P, build_suffix_ids(row.question, reorder_options(row.options, kind)), n)
            recs[row.qid]["noise"][kind] = _finish(map_back(res, n, kind), row)
    del P
    # frame-shift noise
    for kind in NOISE_FRAME_SHIFTS:
        e = encode_frames(fsets[kind], fsets["_meta"][kind])
        P = prefill(full_spec(e))
        for row in rows:
            recs[row.qid]["noise"][kind] = _finish(suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]), row)
        del P, e
    for (m, s), c in comp.items():
        if c["inner"] is None:              # question-independent: cache the prefix
            P = prefill(c)
            outs = {row.qid: suffix_pass(P, sufs[row.qid], recs[row.qid]["n_opt"]) for row in rows}
            del P
        else:                               # FlashVID / HoliTom act inside the LLM and depend on the question
            outs = {row.qid: llm_pass(c, sufs[row.qid], recs[row.qid]["n_opt"]) for row in rows}
        for row in rows:
            res = _finish(outs[row.qid], row)
            res["retention"] = round(c["nvis"] / n_full, 4)
            res["retention_layer_avg"] = round(res["vis_mean_layers"] / n_full, 4)
            recs[row.qid]["methods"].setdefault(m, {})[s] = res
    return [recs[row.qid] for row in rows], fails

n_new, n_err, consec_fail, FIRST = 0, 0, 0, True
def _fail_guard():
    if consec_fail >= 6:
        raise RuntimeError("6 questions failed in a row — stopping instead of wasting GPU hours. Send me the log.")

reader = cf.ThreadPoolExecutor(1)
groups = video_groups()
nxt = next(groups, None)
fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None
while nxt is not None:
    vk, path, g = nxt
    cur_fut = fut
    nxt = next(groups, None) if time_left() > 1200 else None
    if nxt is None and time_left() <= 1200:
        print("⏱️ Time budget reached — stopping after this video")
    fut = reader.submit(read_all_frame_sets, nxt[1]) if nxt else None       # decode the next video meanwhile
    t0 = time.time()
    rows = list(g.itertuples())
    try:
        fsets = cur_fut.result()
        if FIRST:
            equivalence_check(fsets, rows[0])
        recs, fails = run_video(fsets, rows, zlib.crc32(str(vk).encode()), FIRST)
        FIRST = False
    except RuntimeError as e:
        if "check failed" in str(e) or "Equivalence check" in str(e) or "Cache check" in str(e):
            raise
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); _fail_guard(); continue
    except Exception as e:
        n_err += len(rows); consec_fail += len(rows)
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tvideo\t{type(e).__name__}: {e}\n")
        print(f"  ✗ video {vk}: {type(e).__name__}: {e}"); torch.cuda.empty_cache(); _fail_guard(); continue
    for m, s_, err in fails:
        with open(ERR_PATH, "a") as f: f.write(f"{vk}\tcompress {m}@{s_}\t{err}\n")
        print(f"  ! {m}@{s_} failed on video {vk} (skipped for this video): {err}")
    with open(RES_PATH, "a") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
            done.add(r["qid"])
    n_new += len(recs); consec_fail = 0
    for r in recs:
        flips = sum(x["pred"] != r["full"]["pred"] for v in r["methods"].values() for x in v.values())
        nz = "".join("·" if x["pred"] == r["full"]["pred"] else "x" for x in r["noise"].values())
        print(f"  ✓ {r['qid']} [{r['group']}] gt={r['gt']} full={r['full']['pred']} noise[{nz}] "
              f"| {flips}/{sum(len(v) for v in r['methods'].values())} compressed changed")
    print(f"    video done in {time.time() - t0:.0f}s | {len(done)} questions total | {time_left() / 3600:.1f} h left")
    del fsets
    gc.collect(); torch.cuda.empty_cache()
reader.shutdown()
for _d in glob.glob(os.path.join(WORK_DIR, "vmme_*")):
    shutil.rmtree(_d, ignore_errors=True)
print(f"\nFinished: {n_new} new questions, {n_err} errors, {len(done)} total in {RES_PATH}")

### 7 · Per-benchmark analysis

In [ ]:
# ── Per-benchmark analysis (same measures as the main study; H1, H2a, H2b and H3 are recomputed afterwards) ──
import matplotlib.pyplot as plt
R = []
if not os.path.exists(RES_PATH):
    raise SystemExit("No results yet — the run above produced no answers (see its messages / errors.log).")
with open(RES_PATH) as f:
    for line in f:
        try:
            R.append(json.loads(line))
        except Exception:
            pass
R = [r for r in R if len(r.get("noise", {})) == 4]
n = len(R)
if n < 5:
    raise SystemExit(f"Only {n} results — nothing to analyse yet (normal in a quick test with very few questions).")
rng = np.random.default_rng(SEED)
vid = np.array([r["video"] for r in R]); videos = np.unique(vid)
v_rows = [np.flatnonzero(vid == v) for v in videos]
B = 2000
boot = [np.concatenate([v_rows[j] for j in rng.integers(0, len(videos), len(videos))]) for _ in range(B)]
full = np.array([r["full"]["pred"] for r in R]); fcor = np.array([r["full"]["correct"] for r in R])
margin = np.array([r["full"]["margin"] for r in R])
NK = ["frame_shift", "frame_shift_q", "option_cycle", "option_reverse"]
NF = {k: np.array([r["noise"][k]["pred"] for r in R]) != full for k in NK}
STABLE = {k: sum(NF[j] for j in NK if j != k) == 0 for k in NK}       # stable under the other three
DELTA = 0.03

def kl(p_lp, q_lp):
    p_lp, q_lp = np.array(p_lp), np.array(q_lp)
    return float((np.exp(p_lp) * (p_lp - q_lp)).sum())

def pass_arrays(get):
    ok = np.array([get(r) is not None for r in R])
    pred = np.array([get(r)["pred"] if o else "" for r, o in zip(R, ok)])
    cor = np.array([bool(get(r)["correct"]) if o else False for r, o in zip(R, ok)])
    return ok, pred, cor

def excess(flip, ok, idx):
    """Pre-registered Δ: mean over the 4 held-out noise passes of [method flips − held-out noise flips] on the
    questions stable under the other three noise passes (restricted to questions where the method ran)."""
    vals = []
    for k in NK:
        s = STABLE[k][idx] & ok[idx]
        if s.sum() == 0:
            return np.nan
        vals.append(flip[idx][s].mean() - NF[k][idx][s].mean())
    return float(np.mean(vals))

def describe(get, name):
    ok, pred, cor = pass_arrays(get)
    flip = (pred != full) & ok
    idx_all = np.arange(n)
    d = excess(flip, ok, idx_all)
    d_b = np.array([excess(flip, ok, b) for b in boot])
    lo, hi = np.nanpercentile(d_b, [2.5, 97.5])
    h, g = int((fcor & ~cor & ok).sum()), int((~fcor & cor & ok).sum())
    rel_b = [100 * cor[b][ok[b]].mean() / max(fcor[b][ok[b]].mean(), 1e-9) for b in boot]
    return {"pass": name, "coverage": float(ok.mean()), "acc": float(cor[ok].mean()),
            "rel_acc": float(100 * cor[ok].mean() / fcor[ok].mean()),
            "rel_ci": [float(np.percentile(rel_b, 2.5)), float(np.percentile(rel_b, 97.5))],
            "flip": float(flip[ok].mean()), "excess": d, "excess_ci": [float(lo), float(hi)],
            "class": "lossy" if lo > 0 else ("noise-equivalent" if hi < DELTA else "inconclusive"),
            "harmful": h, "helpful": g, "harmful_share": h / (h + g) if h + g else float("nan"),
            "kl": float(np.mean([kl(R[k]["full"]["logp"], get(R[k])["logp"]) for k in np.flatnonzero(ok)])),
            "flops_ratio": float(np.mean([get(R[k])["flops"] for k in np.flatnonzero(ok)]) /
                                 np.mean([R[k]["full"]["flops"] for k in np.flatnonzero(ok)])),
            "retention": float(np.mean([get(R[k]).get("retention", 1.0) for k in np.flatnonzero(ok)])),
            "retention_layer_avg": float(np.mean([get(R[k]).get("retention_layer_avg", get(R[k]).get("retention", 1.0))
                                                 for k in np.flatnonzero(ok)]))}

# ---------------- noise ----------------
fragility = float(np.any(np.stack([NF[k] for k in NK]), 0).mean())
fr_b = [np.any(np.stack([NF[k][b] for k in NK]), 0).mean() for b in boot]
noise_rows = []
for k in NK:
    ok, pred, cor = pass_arrays(lambda r, k=k: r["noise"][k])
    h, g = int((fcor & ~cor).sum()), int((~fcor & cor).sum())
    noise_rows.append({"pass": k, "flip": float(NF[k].mean()), "acc": float(cor.mean()),
                       "heldout_flip_on_stable": float(NF[k][STABLE[k]].mean()), "harmful": h, "helpful": g})
accs = [fcor.mean()] + [x["acc"] for x in noise_rows]
noise_range = 100 * (max(accs) - min(accs))
H_n = sum(x["harmful"] for x in noise_rows); G_n = sum(x["helpful"] for x in noise_rows)
print(f"=== {MODEL} / {BENCHMARK}: {n} questions, {len(videos)} videos | full accuracy {fcor.mean():.3f} ===")
print(f"Fragility (changed by ≥1 of 4 noise passes): {fragility:.3f} "
      f"(95% CI {np.percentile(fr_b, 2.5):.3f}–{np.percentile(fr_b, 97.5):.3f})")
print(f"Noise accuracy range: {noise_range:.1f} points | pooled noise harmful share {H_n / max(H_n + G_n, 1):.2f}")
for x in noise_rows:
    print(f"   {x['pass']:15s} flips {x['flip']:.3f} | held-out flips on stable questions {x['heldout_flip_on_stable']:.3f}")

# ---------------- methods ----------------
rows = []
for m, settings in METHOD_GRID.items():
    for s in settings:
        key = f"K{s}" if m == "dycoke" else str(s)
        rows.append({"method": m, "setting": key,
                     **describe(lambda r, m=m, key=key: r["methods"].get(m, {}).get(key), f"{m}@{key}")})
cells = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
show = cells[["method", "setting", "coverage", "retention", "rel_acc", "flip", "excess", "excess_ci", "class",
              "harmful_share", "kl", "flops_ratio"]].copy()
show["excess_ci"] = show.excess_ci.apply(lambda c: f"{c[0]:+.3f}…{c[1]:+.3f}")
print("\nCells (excess = extra answer changes beyond held-out noise, on noise-stable questions):")
print(show.round(3).to_string(index=False))
cells.to_csv(os.path.join(OUT_DIR, "cells.csv"), index=False)

# ---------------- confidence quartiles ----------------
q = np.digitize(margin, np.quantile(margin, [0.25, 0.5, 0.75]))
noise_any = np.mean(np.stack([NF[k] for k in NK]), 0)
low_keys = [(m, (f"K{s}" if m == "dycoke" else str(s))) for m, v in METHOD_GRID.items() for s in v
            if m != "dycoke" and s <= 0.10]
cf_low = np.mean(np.stack([np.array([(r["methods"].get(m, {}).get(s) or r["full"])["pred"] != r["full"]["pred"]
                                     for r in R]) for m, s in low_keys]), 0)
quart = {"noise": [float(noise_any[q == i].mean()) for i in range(4)],
         "compressed_le10": [float(cf_low[q == i].mean()) for i in range(4)]}
print("\nFlip rate by full-model confidence quartile (Q1 = least sure):", {k: np.round(v, 3).tolist() for k, v in quart.items()})

summary = {"benchmark": BENCHMARK, "study": "replication", "n": n, "n_videos": int(len(videos)), "model": MODEL_ID, "frames": NUM_FRAMES,
           "full_accuracy": float(fcor.mean()), "fragility": fragility,
           "fragility_ci": [float(np.percentile(fr_b, 2.5)), float(np.percentile(fr_b, 97.5))],
           "noise_accuracy_range_points": noise_range, "noise": noise_rows,
           "noise_harmful_share": H_n / max(H_n + G_n, 1), "confidence_quartiles": quart,
           "cells": rows, "time_hours": (time.time() - T_START) / 3600 if "T_START" in dir() else None}
with open(os.path.join(OUT_DIR, f"summary_{BENCHMARK}.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
# ---------------- plot (after saving the numbers, so a plotting problem can never lose them) ----------------
try:
    col = {"flashvid": "#17becf", "fastvid": "#1f77b4", "holitom": "#9467bd", "visionzip": "#d62728",
           "scissor": "#8c564b", "dycoke": "#2ca02c", "uniform": "#7f7f7f", "random": "#ff7f0e"}
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    for m, g in cells.groupby("method", sort=False):
        g = g.sort_values("retention_layer_avg")
        x = 100 * g.retention_layer_avg.values
        ax[0].errorbar(x, g.excess, yerr=[g.excess - g.excess_ci.str[0], g.excess_ci.str[1] - g.excess],
                       fmt="o-", color=col[m], label=m, capsize=2, ms=4)
        ax[1].plot(x, g.rel_acc, "o-", color=col[m], label=m)
    ax[0].axhline(0, c="k", lw=1); ax[0].axhline(DELTA, c="k", lw=0.8, ls=":")
    ax[0].set_xscale("log"); ax[0].set_xlabel("visual tokens kept (%)"); ax[0].set_ylabel("excess flips beyond noise")
    ax[0].set_title(f"{MODEL} / {BENCHMARK}: extra answer changes beyond harmless noise"); ax[0].legend(fontsize=8, ncol=2)
    ax[1].axhline(100, c="k", lw=0.8, ls="--"); ax[1].set_xscale("log")
    ax[1].set_xlabel("visual tokens kept (%)"); ax[1].set_ylabel("relative accuracy (%)"); ax[1].set_title("Relative accuracy")
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"summary_{BENCHMARK}.png"), dpi=150); plt.show()
except Exception as e:
    plt.close("all")
    print(f"(plot skipped: {type(e).__name__}: {e})")

if QUICK_TEST:
    print("\n(Quick test: numbers are meaningless — this only proves the pipeline works.)")
print("Files saved in", OUT_DIR)

## Next
Send the files listed at the top. After the three benchmarks are in for both replication models, H1, H2a, H2b and H3 are recomputed with the main study's rules.